# TripAdvisor Sentiment Analysis Master Notebook

Run these sections in order after uploading `reviews.csv` into the Colab file browser, or after placing the same file at `data/raw/reviews.csv`. The setup cell installs the required R packages and creates the shared helper files used by the workflow.

## Setup Shared Project Files

In [ ]:
# =====================================================================
# Setup Shared Project Files
# =====================================================================
# Google Colab starts with an empty working folder. This cell installs the
# R packages used by the project, then creates the same folders and helper
# files that the local project uses.

required_packages <- c(
  "tidyverse",
  "tidytext",
  "syuzhet",
  "wordcloud",
  "RColorBrewer",
  "lubridate",
  "jsonlite"
)

# Use a dated CRAN snapshot so rerunning the notebook uses stable package versions.
# Colab often preinstalls some packages, so install the full required set from
# the snapshot instead of only installing packages that appear to be missing.
cran_snapshot_url <- "https://packagemanager.posit.co/cran/2026-05-26"
install.packages(required_packages, repos = cran_snapshot_url)

dir.create("scripts", recursive = TRUE, showWarnings = FALSE)
dir.create(file.path("data", "raw"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path("data", "cleaned"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path("output", "figures"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path("output", "reports"), recursive = TRUE, showWarnings = FALSE)

writeLines(
  c(
  "# Shared file paths and source metadata for the hotel review workflow.",
  "# Beginners can think of this file as the project's address book. Instead of",
  "# typing the same folder names in every script, each script loads these shared",
  "# path variables.",
  "",
  "prepared_source_name <- \"Prepared TripAdvisor review CSV\"",
  "",
  "# This project is intentionally a single-property analysis. Step 1 uses this",
  "# name to stop early if someone accidentally supplies a mixed-hotel CSV.",
  "target_hotel_name <- \"Bvlgari Resort Bali\"",
  "",
  "# Input data from the prepared TripAdvisor export.",
  "raw_reviews_path <- file.path(\"data\", \"raw\", \"reviews.csv\")",
  "",
  "# Cleaned and scored datasets created by Steps 2 and 3.",
  "cleaned_reviews_path <- file.path(\"data\", \"cleaned\", \"hotel_cleaned_reviews.csv\")",
  "cleaned_tokens_path <- file.path(\"data\", \"cleaned\", \"hotel_cleaned_tokens.csv\")",
  "sentiment_scores_path <- file.path(\"data\", \"cleaned\", \"hotel_sentiment_scores.csv\")",
  "",
  "# Output folders for charts and CSV summary reports.",
  "figures_dir <- file.path(\"output\", \"figures\")",
  "reports_dir <- file.path(\"output\", \"reports\")",
  "",
  "# Summary outputs created by the trend-monitoring part of Step 4.",
  "annual_review_profile_path <- file.path(reports_dir, \"annual_review_profile.csv\")",
  "sentiment_period_summary_path <- file.path(reports_dir, \"sentiment_period_summary.csv\")",
  "sentiment_drift_monitor_path <- file.path(figures_dir, \"sentiment_drift_monitor.png\")"
  ),
  "scripts/data_config.R"
)

writeLines(
  c(
  "# helpers.R",
  "# Reusable helper functions for TripAdvisor Sentiment Analysis.",
  "# These functions streamline text cleaning, source-column parsing, and plotting.",
  "",
  "library(tidyverse)",
  "library(stringr)",
  "",
  "#' Clean Raw Text for Sentiment Analysis",
  "#'",
  "#' Takes a character vector of raw reviews and cleans it by converting to lowercase,",
  "#' removing HTML elements, preserving word boundaries around Unicode punctuation,",
  "#' removing punctuation, numbers, special symbols, and emojis, and stripping extra",
  "#' white spaces.",
  "#'",
  "#' @param text_vector A character vector of text.",
  "#' @return A cleaned character vector.",
  "#' @export",
  "clean_text <- function(text_vector) {",
  "  if (is.null(text_vector) || length(text_vector) == 0) return(character(0))",
  "",
  "  cleaned <- text_vector %>%",
  "    # 1. Convert to lowercase",
  "    str_to_lower() %>%",
  "    # 2. Remove HTML tags (e.g., <br/>)",
  "    str_replace_all(\"<[^>]*>\", \" \") %>%",
  "    # 3. Remove loose angle brackets that are not real HTML tags.",
  "    str_replace_all(\"[<>]\", \" \") %>%",
  "    # 4. Remove email addresses, URLs, or handles",
  "    str_replace_all(\"http\\\\S+|www\\\\S+\", \"\") %>%",
  "    # 5. Turn accented letters and Unicode punctuation into ASCII when possible.",
  "    # This changes a fullwidth comma into a normal comma before punctuation is",
  "    # removed, so words on both sides stay separate.",
  "    stringi::stri_trans_general(\"Any-Latin; Latin-ASCII\") %>%",
  "    # 6. Replace anything still outside ASCII with a space. This mostly catches",
  "    # emojis and symbols that the English-language sentiment tools cannot score.",
  "    str_replace_all(\"[^\\x01-\\x7F]\", \" \") %>%",
  "    # 7. Remove punctuation and special symbols",
  "    str_replace_all(\"[[:punct:]]\", \" \") %>%",
  "    # 8. Remove numbers",
  "    str_replace_all(\"[[:digit:]]\", \"\") %>%",
  "    # 9. Strip extra white spaces",
  "    str_squish()",
  "",
  "  return(cleaned)",
  "}",
  "",
  "redact_embedded_contact_markers <- function(text_vector) {",
  "  if (is.null(text_vector) || length(text_vector) == 0) return(character(0))",
  "",
  "  text_vector %>%",
  "    str_replace_all(\"https?://\\\\S+|www\\\\.[^\\\\s]+\", \" \") %>%",
  "    str_replace_all(\"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\\\.[A-Za-z]{2,}\", \" \") %>%",
  "    str_replace_all(\"(^|\\\\s)@[A-Za-z0-9_][A-Za-z0-9_.-]*\", \" \") %>%",
  "    str_squish()",
  "}",
  "",
  "normalize_trip_type <- function(trip_type_vector) {",
  "  if (is.null(trip_type_vector) || length(trip_type_vector) == 0) return(character(0))",
  "",
  "  trip_type_vector %>%",
  "    # TripAdvisor can append source-disclosure text after the traveler type.",
  "    # Keep the traveler type, but remove the disclosure so grouping stays clean.",
  "    str_replace_all(regex(\"\\\\s*review collected in partnership with.*$\", ignore_case = TRUE), \"\") %>%",
  "    str_replace_all(regex(\"\\\\s*this business uses tools provided by tripadvisor.*$\", ignore_case = TRUE), \"\") %>%",
  "    str_squish()",
  "}",
  "",
  "#' Remove Stopwords from Cleaned Text",
  "#'",
  "#' Removes common English words (stop words) that don't carry sentiment.",
  "#' Uses tidytext package internally.",
  "#'",
  "#' @param text_df A tibble/dataframe containing the tokenized words.",
  "#' @param word_column The name of the column containing words (as a symbol or character).",
  "#' @return A tibble with stopwords removed.",
  "#' @export",
  "remove_stopwords <- function(text_df, word_column = \"word\") {",
  "  if (!requireNamespace(\"tidytext\", quietly = TRUE)) {",
  "    stop(\"Package 'tidytext' is required. Please install it first.\")",
  "  }",
  "",
  "  data(\"stop_words\", package = \"tidytext\")",
  "",
  "  # Remove standard English stopwords",
  "  text_df %>%",
  "    anti_join(stop_words, by = setNames(\"word\", word_column))",
  "}",
  "",
  "#' Normalize Slang and Expand Contractions (Common TripAdvisor Terms)",
  "#'",
  "#' A custom dictionary lookup to clean common abbreviations or slang found in TripAdvisor reviews.",
  "#'",
  "#' @param text_vector A character vector of text.",
  "#' @return A normalized character vector.",
  "#' @export",
  "normalize_slang <- function(text_vector) {",
  "  # Review sites often use curly apostrophes, such as don't with a smart quote.",
  "  # Change those apostrophes to a plain apostrophe before matching contractions.",
  "  smart_apostrophe_pattern <- paste0(",
  "    \"[\",",
  "    intToUtf8(c(0x2018, 0x2019, 0x201B, 0x2032, 0x02BC, 0xFF07, 0x00B4)),",
  "    \"`]\"",
  "  )",
  "",
  "  replacements <- c(",
  "    \"\\\\bcan\\\\s*'?t\\\\b\" = \"cannot\",",
  "    \"\\\\bwon\\\\s*'?t\\\\b\" = \"will not\",",
  "    \"\\\\bdon\\\\s*'?t\\\\b\" = \"do not\",",
  "    \"\\\\bdoesn\\\\s*'?t\\\\b\" = \"does not\",",
  "    \"\\\\bdidn\\\\s*'?t\\\\b\" = \"did not\",",
  "    \"\\\\bisn\\\\s*'?t\\\\b\" = \"is not\",",
  "    \"\\\\baren\\\\s*'?t\\\\b\" = \"are not\",",
  "    \"\\\\bwasn\\\\s*'?t\\\\b\" = \"was not\",",
  "    \"\\\\bweren\\\\s*'?t\\\\b\" = \"were not\",",
  "    \"\\\\bwouldn\\\\s*'?t\\\\b\" = \"would not\",",
  "    \"\\\\bcouldn\\\\s*'?t\\\\b\" = \"could not\",",
  "    \"\\\\bshouldn\\\\s*'?t\\\\b\" = \"should not\",",
  "    \"\\\\bw\\\\s*/\\\\s*o\\\\b\" = \"without\",",
  "    \"\\\\bw\\\\s*/(?=\\\\s|$)\" = \"with\",",
  "    \"\\\\bsooo+\\\\b\" = \"so\",",
  "    \"\\\\bgooo+d\\\\b\" = \"good\",",
  "    \"\\\\blooo+ve\\\\b\" = \"love\",",
  "    \"\\\\bawesome\\\\b\" = \"excellent\",",
  "    \"\\\\bbtwn\\\\b\" = \"between\",",
  "    \"\\\\bpls\\\\b\" = \"please\",",
  "    \"\\\\bvry\\\\b\" = \"very\",",
  "    \"\\\\bgrt\\\\b\" = \"great\",",
  "    \"\\\\bu\\\\b\" = \"you\"",
  "  )",
  "  ",
  "  cleaned <- str_replace_all(text_vector, smart_apostrophe_pattern, \"'\")",
  "  for (pattern in names(replacements)) {",
  "    cleaned <- str_replace_all(cleaned, regex(pattern, ignore_case = TRUE), replacements[pattern])",
  "  }",
  "  return(cleaned)",
  "}",
  "",
  "score_negation_aware_sentiment <- function(text_vector, method = \"afinn\", negation_window = 3) {",
  "  if (!requireNamespace(\"syuzhet\", quietly = TRUE)) {",
  "    stop(\"Package 'syuzhet' is required. Please install it first.\", call. = FALSE)",
  "  }",
  "",
  "  if (is.null(text_vector) || length(text_vector) == 0) return(numeric(0))",
  "",
  "  negation_words <- c(\"no\", \"not\", \"never\", \"without\", \"cannot\")",
  "  contrast_words <- c(\"but\", \"however\", \"though\", \"although\", \"yet\")",
  "",
  "  score_one_review <- function(text_value) {",
  "    if (is.na(text_value)) {",
  "      return(0)",
  "    }",
  "",
  "    review_words <- str_split(str_squish(as.character(text_value)), \"\\\\s+\")[[1]]",
  "    review_words <- review_words[review_words != \"\"]",
  "",
  "    if (length(review_words) == 0) {",
  "      return(0)",
  "    }",
  "",
  "    word_scores <- syuzhet::get_sentiment(review_words, method = method)",
  "    negated_positions <- rep(FALSE, length(review_words))",
  "    negation_positions <- which(review_words %in% negation_words)",
  "",
  "    for (position in negation_positions) {",
  "      if (position == length(review_words)) {",
  "        next",
  "      }",
  "",
  "      window_positions <- seq(position + 1, min(length(review_words), position + negation_window))",
  "      first_contrast <- which(review_words[window_positions] %in% contrast_words)",
  "",
  "      if (length(first_contrast) > 0) {",
  "        window_positions <- window_positions[seq_len(first_contrast[1] - 1)]",
  "      }",
  "",
  "      if (length(window_positions) > 0) {",
  "        negated_positions[window_positions] <- TRUE",
  "      }",
  "    }",
  "",
  "    scored_negated_words <- negated_positions & !is.na(word_scores) & word_scores != 0",
  "    word_scores[scored_negated_words] <- -word_scores[scored_negated_words]",
  "",
  "    sum(word_scores, na.rm = TRUE)",
  "  }",
  "",
  "  vapply(text_vector, score_one_review, numeric(1))",
  "}",
  "",
  "# Count how many words are in each cleaned review.",
  "# Example: \"great service and food\" has 4 words.",
  "count_cleaned_words <- function(text_vector) {",
  "  if (is.null(text_vector) || length(text_vector) == 0) return(integer(0))",
  "",
  "  # Cleaned reviews use spaces between words. Counting non-empty chunks separated",
  "  # by spaces gives the review length used for sentiment normalization.",
  "  text_values <- as.character(text_vector)",
  "  text_values[is.na(text_values)] <- \"\"",
  "  text_values <- str_squish(text_values)",
  "",
  "  if_else(",
  "    text_values == \"\",",
  "    0L,",
  "    as.integer(str_count(text_values, \"\\\\S+\"))",
  "  )",
  "}",
  "",
  "# Rescale raw sentiment scores so reviews can be compared at the same length.",
  "# Example: if a 200-word review has an AFINN score of 20 and the target length",
  "# is 100 words, the normalized score is 10.",
  "normalize_score_to_word_count <- function(score_values, word_counts, target_word_count) {",
  "  # Raw lexicon scores are sums, so longer reviews can have larger totals just",
  "  # because they contain more words. This rescales each score to the same review",
  "  # length while preserving the original raw score in a separate column.",
  "  normalized_scores <- rep(NA_real_, length(score_values))",
  "  valid_rows <- !is.na(score_values) &",
  "    !is.na(word_counts) &",
  "    word_counts > 0 &",
  "    !is.na(target_word_count) &",
  "    target_word_count > 0",
  "",
  "  normalized_scores[valid_rows] <- score_values[valid_rows] / word_counts[valid_rows] * target_word_count",
  "  normalized_scores",
  "}",
  "",
  "# Calculate an average after removing the most extreme low and high values.",
  "# With trim = 0.1, R removes the lowest 10% and highest 10% before averaging.",
  "calculate_trimmed_mean <- function(values, trim = 0.1) {",
  "  # A trimmed mean drops the lowest and highest tails before averaging. It is",
  "  # useful when one unusual review should not control the period summary.",
  "  complete_values <- values[!is.na(values)]",
  "  if (length(complete_values) == 0) {",
  "    return(NA_real_)",
  "  }",
  "",
  "  mean(complete_values, trim = trim)",
  "}",
  "",
  "# Measure normal variation with MAD, a robust alternative to standard deviation.",
  "calculate_robust_mad <- function(values, minimum_count = 20) {",
  "  # MAD means median absolute deviation. It estimates normal variation around",
  "  # the median and is less sensitive to outliers than a standard deviation.",
  "  complete_values <- values[!is.na(values)]",
  "  if (length(complete_values) < minimum_count) {",
  "    return(NA_real_)",
  "  }",
  "",
  "  mad_value <- stats::mad(",
  "    complete_values,",
  "    center = stats::median(complete_values),",
  "    constant = 1.4826,",
  "    na.rm = TRUE",
  "  )",
  "",
  "  if (is.na(mad_value) || mad_value == 0) {",
  "    return(NA_real_)",
  "  }",
  "",
  "  mad_value",
  "}",
  "",
  "# Compare one current value with earlier historical values.",
  "# A result near 0 means \"typical\"; below -2 means \"unusually low\" by this rule.",
  "calculate_robust_z <- function(value, baseline_values, minimum_count = 20) {",
  "  # A robust z-score asks how far the current value is from the historical",
  "  # median, measured in MAD units. Negative values mean the current period is",
  "  # below the historical baseline.",
  "  baseline_values <- baseline_values[!is.na(baseline_values)]",
  "  mad_value <- calculate_robust_mad(baseline_values, minimum_count = minimum_count)",
  "",
  "  if (length(value) == 0 || is.na(value[[1]]) || is.na(mad_value)) {",
  "    return(NA_real_)",
  "  }",
  "",
  "  (value[[1]] - stats::median(baseline_values)) / mad_value",
  "}",
  "",
  "# Prepare raw and length-normalized sentiment columns in one consistent way.",
  "# Step 4 and Step 5 both call this helper so their charts and CSV files use the",
  "# same word-count denominator and cannot accidentally drift apart.",
  "prepare_length_normalized_sentiment <- function(data) {",
  "  required_columns <- c(\"cleaned_text\", \"score_syuzhet\", \"score_afinn\")",
  "  missing_columns <- setdiff(required_columns, names(data))",
  "  if (length(missing_columns) > 0) {",
  "    stop(",
  "      paste(\"Missing columns needed for length-normalized sentiment:\", paste(missing_columns, collapse = \", \")),",
  "      call. = FALSE",
  "    )",
  "  }",
  "",
  "  # Convert the raw score columns to numbers and create a word count if Step 3",
  "  # has not already saved one. Later scripts call this helper so every chart and",
  "  # table uses the same normalization rule.",
  "  normalized_data <- data %>%",
  "    mutate(",
  "      score_syuzhet_raw = readr::parse_number(as.character(score_syuzhet)),",
  "      score_afinn_raw = readr::parse_number(as.character(score_afinn)),",
  "      review_word_count = if (\"review_word_count\" %in% names(data)) {",
  "        readr::parse_number(as.character(review_word_count))",
  "      } else {",
  "        count_cleaned_words(cleaned_text)",
  "      }",
  "    )",
  "",
  "  if (\"median_review_word_count\" %in% names(normalized_data)) {",
  "    # If Step 3 already saved the median review length, reuse it. That keeps",
  "    # later scripts aligned with the scored CSV.",
  "    possible_target <- readr::parse_number(as.character(normalized_data$median_review_word_count))",
  "    target_word_count <- stats::median(possible_target[possible_target > 0], na.rm = TRUE)",
  "  } else {",
  "    # If a small test fixture skipped Step 3, calculate the median length here.",
  "    target_word_count <- stats::median(normalized_data$review_word_count[normalized_data$review_word_count > 0], na.rm = TRUE)",
  "  }",
  "",
  "  if (is.na(target_word_count)) {",
  "    stop(\"No cleaned words were found, so sentiment scores cannot be length-normalized.\", call. = FALSE)",
  "  }",
  "",
  "  normalized_data %>%",
  "    mutate(",
  "      median_review_word_count = target_word_count,",
  "      # These two columns answer: what would the score look like if every review",
  "      # were the same length as the typical review in this dataset?",
  "      score_syuzhet_per_median_review = normalize_score_to_word_count(",
  "        score_syuzhet_raw,",
  "        review_word_count,",
  "        target_word_count",
  "      ),",
  "      score_afinn_per_median_review = normalize_score_to_word_count(",
  "        score_afinn_raw,",
  "        review_word_count,",
  "        target_word_count",
  "      )",
  "    )",
  "}",
  "",
  "#' Standardize Raw Hotel Review Columns",
  "#'",
  "#' Review exports often use source-specific column names such as `text`,",
  "#' `ratings.Overall`, and `date_stayed`. This function maps common variants into",
  "#' the columns used by the rest of the workflow. Extra source fields are kept",
  "#' only when they are useful for analysis and do not expose response metadata,",
  "#' source URLs, or direct reviewer identifiers. Reviewer location is kept as",
  "#' geographic context because the project only uses it in aggregate tables.",
  "#'",
  "#' @param raw_data A dataframe read from the raw reviews CSV.",
  "#' @return A tibble with standardized columns plus any extra source metadata.",
  "#' @export",
  "standardize_hotel_reviews <- function(raw_data) {",
  "  if (is.null(raw_data) || nrow(raw_data) == 0) {",
  "    stop(\"The raw review dataset is empty.\", call. = FALSE)",
  "  }",
  "  ",
  "  normalize_source_column_names <- function(column_names) {",
  "    column_names %>%",
  "      str_replace_all(\"([a-z0-9])([A-Z])\", \"\\\\1_\\\\2\") %>%",
  "      str_to_lower() %>%",
  "      str_replace_all(\"[^a-z0-9]+\", \"_\") %>%",
  "      str_replace_all(\"^_|_$\", \"\")",
  "  }",
  "",
  "  normalized_names <- names(raw_data) %>%",
  "    normalize_source_column_names()",
  "  ",
  "  find_column <- function(candidates, required = FALSE) {",
  "    normalized_candidates <- candidates %>%",
  "      normalize_source_column_names()",
  "    ",
  "    match_index <- match(normalized_candidates, normalized_names, nomatch = 0)",
  "    match_index <- match_index[match_index > 0]",
  "    ",
  "    if (length(match_index) > 0) {",
  "      return(names(raw_data)[match_index[1]])",
  "    }",
  "    ",
  "    if (required) {",
  "      stop(",
  "        paste(",
  "          \"Could not find a required review-text column. Available columns:\",",
  "          paste(names(raw_data), collapse = \", \")",
  "        ),",
  "        call. = FALSE",
  "      )",
  "    }",
  "    ",
  "    NA_character_",
  "  }",
  "  ",
  "  id_col <- find_column(c(\"review_id\", \"id\", \"reviews_id\"))",
  "  hotel_col <- find_column(c(\"hotel_name\", \"offering_name\", \"property_name\", \"listing_name\"))",
  "  title_col <- find_column(c(\"title\", \"review_title\", \"reviews_title\", \"heading\", \"headline\"))",
  "  review_col <- find_column(",
  "    c(\"review_text\", \"review\", \"reviews_text\", \"text\", \"content\", \"body\", \"comment\", \"comments\"),",
  "    required = TRUE",
  "  )",
  "  rating_col <- find_column(",
  "    c(\"rating\", \"ratings_overall\", \"ratings.overall\", \"overall_rating\", \"review_rating\", \"reviewer_score\", \"score\", \"stars\")",
  "  )",
  "  date_col <- find_column(",
  "    c(\"review_date\", \"date\", \"date_stayed\", \"date_of_stay\", \"reviews_date\", \"reviewed_at\", \"created_at\", \"published_date\")",
  "  )",
  "  ",
  "  optional_character_column <- function(column_name) {",
  "    if (is.na(column_name)) {",
  "      rep(NA_character_, nrow(raw_data))",
  "    } else {",
  "      as.character(raw_data[[column_name]])",
  "    }",
  "  }",
  "  ",
  "  optional_rating_column <- function(column_name) {",
  "    if (is.na(column_name)) {",
  "      rep(NA_real_, nrow(raw_data))",
  "    } else {",
  "      readr::parse_number(as.character(raw_data[[column_name]]))",
  "    }",
  "  }",
  "  ",
  "  standardized <- tibble(",
  "    review_id = if (is.na(id_col)) seq_len(nrow(raw_data)) else as.character(raw_data[[id_col]]),",
  "    hotel_name = optional_character_column(hotel_col),",
  "    title = optional_character_column(title_col),",
  "    review_text = as.character(raw_data[[review_col]]),",
  "    rating = optional_rating_column(rating_col),",
  "    review_date = optional_character_column(date_col)",
  "  )",
  "",
  "  mapped_columns <- na.omit(c(id_col, hotel_col, title_col, review_col, rating_col, date_col))",
  "  extra_columns <- setdiff(names(raw_data), mapped_columns)",
  "  normalized_name_lookup <- setNames(normalized_names, names(raw_data))",
  "",
  "  # Direct reviewer identifiers are not needed for this aggregate analysis.",
  "  # Dropping them keeps generated, tracked datasets focused on the review",
  "  # content and ratings instead of personal profile details. Reviewer location",
  "  # is allowed because it supports aggregate geography summaries.",
  "  direct_reviewer_identifier_columns <- c(",
  "    \"author\",",
  "    \"reviewer\",",
  "    \"reviewer_id\",",
  "    \"reviewer_name\",",
  "    \"name\",",
  "    \"user_id\",",
  "    \"user_name\",",
  "    \"username\",",
  "    \"author_id\",",
  "    \"author_name\",",
  "    \"member_id\",",
  "    \"profile_id\",",
  "    \"profile_url\",",
  "    \"reviewer_profile_url\",",
  "    \"user_profile_url\"",
  "  )",
  "  allowed_reviewer_geography_columns <- c(",
  "    \"reviewer_location\"",
  "  )",
  "  normalized_allowed_reviewer_geography_columns <- normalize_source_column_names(allowed_reviewer_geography_columns)",
  "  forbidden_source_metadata_columns <- c(",
  "    \"reviewer_helpful_votes\",",
  "    \"machine_translated\",",
  "    \"show_original_available\",",
  "    \"has_management_response\",",
  "    \"review_url\",",
  "    \"page_url\",",
  "    \"source_observed_at\",",
  "    \"source_snapshot_id\"",
  "  )",
  "  forbidden_source_metadata_pattern <- paste(",
  "    c(",
  "      \"^management_response($|_)\",",
  "      \"(^|_)url$\"",
  "    ),",
  "    collapse = \"|\"",
  "  )",
  "  direct_reviewer_identifier_pattern <- paste(",
  "    c(",
  "      \"(^|_)reviewer_?id$\",",
  "      \"^user_?id$\",",
  "      \"^author_?id$\",",
  "      \"^member_?id$\",",
  "      \"^profile_?(id|url)$\",",
  "      \"^(reviewer|user|author)_profile_?(id|url)$\"",
  "    ),",
  "    collapse = \"|\"",
  "  )",
  "  normalized_extra_names <- normalized_name_lookup[extra_columns]",
  "  compact_extra_names <- str_replace_all(normalized_extra_names, \"_\", \"\")",
  "  compact_direct_reviewer_identifier_columns <- str_replace_all(",
  "    direct_reviewer_identifier_columns,",
  "    \"_\",",
  "    \"\"",
  "  )",
  "  compact_forbidden_source_metadata_columns <- str_replace_all(",
  "    forbidden_source_metadata_columns,",
  "    \"_\",",
  "    \"\"",
  "  )",
  "  is_direct_reviewer_identifier <- normalized_extra_names %in% direct_reviewer_identifier_columns |",
  "    compact_extra_names %in% compact_direct_reviewer_identifier_columns |",
  "    str_detect(normalized_extra_names, direct_reviewer_identifier_pattern) |",
  "    str_detect(compact_extra_names, \"^(reviewer|user|author|member|profile).*(id|name|location|url)$\")",
  "  is_allowed_reviewer_geography <- normalized_extra_names %in% normalized_allowed_reviewer_geography_columns",
  "  is_forbidden_source_metadata <- normalized_extra_names %in% forbidden_source_metadata_columns |",
  "    compact_extra_names %in% compact_forbidden_source_metadata_columns |",
  "    str_detect(normalized_extra_names, forbidden_source_metadata_pattern) |",
  "    str_detect(compact_extra_names, \"^managementresponse|url$\")",
  "  extra_columns <- extra_columns[",
  "    (!is_direct_reviewer_identifier | is_allowed_reviewer_geography) & !is_forbidden_source_metadata",
  "  ]",
  "",
  "  if (length(extra_columns) > 0) {",
  "    # Keep reviewer geography under one canonical column name. This lets",
  "    # exports such as reviewerLocation or \"Reviewer Location\" feed the same",
  "    # annual profile code as reviewer_location.",
  "    extra_data <- raw_data[extra_columns]",
  "    allowed_geography_extra_columns <- extra_columns[",
  "      normalized_name_lookup[extra_columns] %in% normalized_allowed_reviewer_geography_columns",
  "    ]",
  "",
  "    if (length(allowed_geography_extra_columns) > 0) {",
  "      reviewer_location_values <- purrr::reduce(",
  "        raw_data[allowed_geography_extra_columns],",
  "        dplyr::coalesce",
  "      )",
  "      extra_data <- extra_data[setdiff(names(extra_data), allowed_geography_extra_columns)]",
  "      extra_data[[\"reviewer_location\"]] <- as.character(reviewer_location_values)",
  "    }",
  "",
  "    standardized <- bind_cols(standardized, extra_data)",
  "  }",
  "",
  "  standardized <- standardized %>%",
  "    mutate(across(where(is.character), redact_embedded_contact_markers)) %>%",
  "    mutate(",
  "      across(c(review_id, hotel_name, title, review_text, review_date), str_squish)",
  "    ) %>%",
  "    filter(!is.na(review_text), review_text != \"\")",
  "",
  "  if (\"trip_type\" %in% names(standardized)) {",
  "    standardized <- standardized %>%",
  "      mutate(trip_type = normalize_trip_type(as.character(trip_type)))",
  "  }",
  "  ",
  "  if (nrow(standardized) == 0) {",
  "    stop(\"No non-empty review text was found after standardizing the raw dataset.\", call. = FALSE)",
  "  }",
  "  ",
  "  standardized",
  "}"
  ),
  "scripts/helpers.R"
)

cat("Colab setup complete. Upload reviews.csv before running Step 1, or place the same file at data/raw/reviews.csv.\n")

## Step 1: Import Review Data

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 1: Import Review Data
# =====================================================================
# PURPOSE OF THIS SCRIPT:
# Use the prepared TripAdvisor review CSV provided with this project.

source("scripts/data_config.R")

required_columns <- c(
  "review_id",
  "hotel_name",
  "title",
  "review_text",
  "rating",
  "review_date",
  "stay_date",
  "trip_type"
)

direct_reviewer_identifier_columns <- c(
  "author",
  "reviewer",
  "reviewer_id",
  "reviewer_name",
  "name",
  "user_id",
  "user_name",
  "username",
  "author_id",
  "author_name",
  "member_id",
  "profile_id",
  "profile_url",
  "reviewer_profile_url",
  "user_profile_url"
)

allowed_reviewer_geography_columns <- c(
  "reviewer_location"
)

forbidden_source_metadata_columns <- c(
  "reviewer_helpful_votes",
  "machine_translated",
  "show_original_available",
  "has_management_response",
  "review_url",
  "page_url",
  "source_observed_at",
  "source_snapshot_id"
)

embedded_contact_pattern <- paste(
  c(
    "https?://\\S+",
    "www\\.[^\\s]+",
    "[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}",
    "(^|\\s)@[A-Za-z0-9_][A-Za-z0-9_.-]*"
  ),
  collapse = "|"
)

source_disclosure_pattern <- paste(
  c(
    "review collected in partnership",
    "official review collection partners",
    "this business uses tools provided by tripadvisor"
  ),
  collapse = "|"
)

normalize_column_name <- function(column_names) {
  # Column names can arrive as snake_case, camelCase, or names with spaces.
  # This turns all of those into the same simple shape before validation.
  normalized <- gsub("([a-z0-9])([A-Z])", "\\1_\\2", column_names, perl = TRUE)
  normalized <- tolower(normalized)
  normalized <- gsub("[^a-z0-9]+", "_", normalized)
  gsub("^_|_$", "", normalized)
}

find_direct_reviewer_identifier_columns <- function(column_names) {
  normalized_names <- normalize_column_name(column_names)
  compact_names <- gsub("_", "", normalized_names)
  normalized_forbidden <- normalize_column_name(direct_reviewer_identifier_columns)
  compact_forbidden <- gsub("_", "", normalized_forbidden)
  normalized_allowed_geography <- normalize_column_name(allowed_reviewer_geography_columns)
  direct_identifier_pattern <- paste(
    c(
      "(^|_)reviewer_?id$",
      "^user_?id$",
      "^author_?id$",
      "^member_?id$",
      "^profile_?(id|url)$",
      "^(reviewer|user|author)_profile_?(id|url)$"
    ),
    collapse = "|"
  )

  is_direct_identifier <- normalized_names %in% normalized_forbidden |
    compact_names %in% compact_forbidden |
    grepl(direct_identifier_pattern, normalized_names, perl = TRUE) |
    grepl("^(reviewer|user|author|member|profile).*(id|name|location|url)$", compact_names, perl = TRUE)
  is_allowed_geography <- normalized_names %in% normalized_allowed_geography
  is_forbidden <- is_direct_identifier & !is_allowed_geography

  column_names[is_forbidden]
}

find_forbidden_source_metadata_columns <- function(column_names) {
  normalized_names <- normalize_column_name(column_names)
  compact_names <- gsub("_", "", normalized_names)
  normalized_forbidden <- normalize_column_name(forbidden_source_metadata_columns)
  compact_forbidden <- gsub("_", "", normalized_forbidden)
  forbidden_source_metadata_pattern <- paste(
    c(
      "^management_response($|_)",
      "(^|_)url$"
    ),
    collapse = "|"
  )

  is_forbidden <- normalized_names %in% normalized_forbidden |
    compact_names %in% compact_forbidden |
    grepl(forbidden_source_metadata_pattern, normalized_names, perl = TRUE) |
    grepl("^managementresponse|url$", compact_names, perl = TRUE)

  column_names[is_forbidden]
}

find_embedded_contact_columns <- function(reviews) {
  # URLs, emails, and social handles are not needed for this aggregate analysis.
  # Every uploaded column is scanned before the file can replace tracked raw data.
  scanned_columns <- names(reviews)
  has_contact_markers <- vapply(
    scanned_columns,
    function(column_name) {
      values <- as.character(reviews[[column_name]])
      values[is.na(values)] <- ""
      any(grepl(embedded_contact_pattern, values, perl = TRUE, ignore.case = TRUE))
    },
    logical(1)
  )

  scanned_columns[has_contact_markers]
}

find_source_disclosure_columns <- function(reviews) {
  # TripAdvisor sometimes adds collection-source notices to the trip type text.
  # The analysis needs the guest trip context, not those platform disclosure strings.
  checked_columns <- intersect("trip_type", names(reviews))
  has_source_disclosures <- vapply(
    checked_columns,
    function(column_name) {
      values <- as.character(reviews[[column_name]])
      values[is.na(values)] <- ""
      any(grepl(source_disclosure_pattern, values, perl = TRUE, ignore.case = TRUE))
    },
    logical(1)
  )

  checked_columns[has_source_disclosures]
}

read_review_csv <- function(csv_path) {
  utils::read.csv(
    csv_path,
    check.names = FALSE,
    stringsAsFactors = FALSE,
    fileEncoding = "UTF-8"
  )
}

validate_prepared_reviews <- function(csv_path = raw_reviews_path, announce = TRUE) {
  if (!file.exists(csv_path) || file.info(csv_path)$size == 0) {
    stop(
      paste(
        "Missing prepared review CSV at:",
        csv_path,
        "Place the provided CSV at this path before running the workflow.",
        sep = "\n"
      ),
      call. = FALSE
    )
  }

  reviews <- read_review_csv(csv_path)
  missing_columns <- setdiff(required_columns, names(reviews))

  if (length(missing_columns) > 0) {
    stop(
      paste(
        "The raw review CSV is missing required columns:",
        paste(missing_columns, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  direct_identifier_columns <- find_direct_reviewer_identifier_columns(names(reviews))
  if (length(direct_identifier_columns) > 0) {
    stop(
      paste(
        "The raw review CSV includes direct reviewer identifier columns.",
        "Remove these columns before accepting or tracking the raw file:",
        paste(direct_identifier_columns, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  forbidden_metadata_columns <- find_forbidden_source_metadata_columns(names(reviews))
  if (length(forbidden_metadata_columns) > 0) {
    stop(
      paste(
        "The raw review CSV includes forbidden source metadata columns.",
        "Remove these columns before accepting or tracking the raw file:",
        paste(forbidden_metadata_columns, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  source_disclosure_columns <- find_source_disclosure_columns(reviews)
  if (length(source_disclosure_columns) > 0) {
    stop(
      paste(
        "The raw review CSV includes source-disclosure text in trip_type values.",
        "Remove TripAdvisor collection disclosures before accepting or tracking the raw file.",
        "Affected columns:",
        paste(source_disclosure_columns, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  contact_columns <- find_embedded_contact_columns(reviews)
  if (length(contact_columns) > 0) {
    stop(
      paste(
        "The raw review CSV includes embedded contact markers in raw columns.",
        "Remove URLs, email addresses, and social handles before accepting or tracking the raw file.",
        "Affected columns:",
        paste(contact_columns, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  if (nrow(reviews) == 0) {
    stop("The raw review CSV has no rows.", call. = FALSE)
  }

  full_reviews <- reviews[!is.na(reviews$review_text) & trimws(reviews$review_text) != "", ]

  if (nrow(full_reviews) == 0) {
    stop(
      "The raw review CSV has rows, but no usable review text.",
      call. = FALSE
    )
  }

  hotel_names <- trimws(as.character(full_reviews$hotel_name))
  blank_hotel_rows <- is.na(hotel_names) | hotel_names == ""

  if (any(blank_hotel_rows)) {
    stop(
      paste(
        "The raw review CSV has full review rows with blank hotel_name values.",
        "Every analyzed row must identify the hotel as:",
        target_hotel_name
      ),
      call. = FALSE
    )
  }

  hotel_names <- unique(hotel_names)

  if (length(hotel_names) == 0) {
    stop(
      paste(
        "The raw review CSV must identify the hotel as:",
        target_hotel_name
      ),
      call. = FALSE
    )
  }

  unexpected_hotel_names <- setdiff(hotel_names, target_hotel_name)
  if (length(unexpected_hotel_names) > 0) {
    stop(
      paste(
        "This workflow is prepared for Bvlgari Resort Bali only.",
        "Unexpected hotel_name values:",
        paste(unexpected_hotel_names, collapse = ", ")
      ),
      call. = FALSE
    )
  }

  if (isTRUE(announce)) {
    cat("Raw review rows available:", nrow(reviews), "\n")
    cat("Rows with review text:", nrow(full_reviews), "\n")
    cat("Hotel validated:", target_hotel_name, "\n")
  }

  invisible(csv_path)
}

copy_root_uploaded_reviews <- function() {
  # In Google Colab, beginners often upload reviews.csv into the notebook root.
  # Validate that file before copying so a stray upload cannot replace the
  # prepared Bvlgari CSV with a wrong hotel, private contacts, or source
  # metadata that should not be tracked.
  if (!file.exists("reviews.csv")) {
    return(invisible(FALSE))
  }

  validate_prepared_reviews("reviews.csv", announce = FALSE)
  dir.create(dirname(raw_reviews_path), recursive = TRUE, showWarnings = FALSE)

  if (!file.copy("reviews.csv", raw_reviews_path, overwrite = TRUE)) {
    stop("Could not copy reviews.csv into data/raw/reviews.csv.", call. = FALSE)
  }

  invisible(TRUE)
}

copy_root_uploaded_reviews()

cat("Dataset source:", prepared_source_name, "\n")
cat("Expected raw CSV:", raw_reviews_path, "\n")

validate_prepared_reviews()

## Step 2: Data Cleaning and Preprocessing

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 2: Data Cleaning & Preprocessing
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# Text written by humans is very "messy". We use emojis, punctuation like "!!!",
# and slang like "u" instead of "you". Before a computer can understand emotions, 
# we have to scrub all this "noise" away. This is called Text Pre-Processing.

In [ ]:
# =====================================================================
# STEP 1: Load Required Packages (Adding Tools)
# =====================================================================
# 'tidyverse' helps us filter and change table data easily.
library(tidyverse)
# 'tidytext' is a specialized toolset used purely for analyzing text and words.
library(tidytext)

# We wrote a file called 'helpers.R' that contains our secret cleaning recipes.
# 'source()' tells the computer to load our custom recipes into memory.
source("scripts/data_config.R")
source("scripts/helpers.R")

cat("Helper functions and libraries loaded!\n")

In [ ]:
# =====================================================================
# STEP 2: Open the Raw Data
# =====================================================================
# 'read_csv' opens the prepared reviews CSV that Step 1 validated.
if (!file.exists(raw_reviews_path)) {
  stop(
    paste(
      "Missing raw review data at:",
      raw_reviews_path,
      "Run Step 1 first with: Rscript 01_data_import.R"
    ),
    call. = FALSE
  )
}

raw_data <- read_csv(raw_reviews_path, show_col_types = FALSE, guess_max = 10000)

sample_size <- suppressWarnings(as.integer(Sys.getenv("HOTEL_REVIEW_SAMPLE_SIZE", "0")))
if (!is.na(sample_size) && sample_size > 0 && nrow(raw_data) > sample_size) {
  raw_data <- raw_data %>% slice_head(n = sample_size)
  cat("Using the first", sample_size, "reviews because HOTEL_REVIEW_SAMPLE_SIZE is set.\n")
}

raw_data <- standardize_hotel_reviews(raw_data)

cat("Successfully loaded", nrow(raw_data), "raw reviews!\n")

In [ ]:
# =====================================================================
# STEP 3: The Cleaning Pipeline
# =====================================================================
# The %>% symbol is a "pipe". It means "AND THEN".
# We are taking the raw data, AND THEN changing (mutating) the text column.
# 
# Our custom 'normalize_slang' recipe fixes internet slang (e.g., sooo -> so).
# We run it before 'clean_text' so abbreviations with punctuation, such as
# "w/" and "w/o", can be expanded before the slash is removed.
# Our custom 'clean_text' recipe removes numbers, exclamation marks, and emojis.

cleaned_reviews_df <- raw_data %>%
  mutate(
    # Create a new column called 'cleaned_text' that contains the scrubbed reviews.
    cleaned_text = normalize_slang(review_text),
    cleaned_text = clean_text(cleaned_text)
  )

# Let's print out the 2nd row to see what the computer did!
cat("--- THIS WAS THE MESSY ORIGINAL TEXT ---\n", cleaned_reviews_df$review_text[2], "\n\n")
cat("--- THIS IS THE PERFECTLY CLEANED TEXT ---\n", cleaned_reviews_df$cleaned_text[2], "\n")

In [ ]:
# =====================================================================
# STEP 4: Tokenization (Chopping Sentences into Words)
# =====================================================================
# Computers don't read paragraphs; they read individual words.
# 'unnest_tokens' is a tool that takes a full sentence and chops it up.
# If a review is 10 words long, this tool creates 10 separate rows in our table!
# This process is formally called "Tokenization".

tokenized_df <- cleaned_reviews_df %>%
  unnest_tokens(output = word, input = cleaned_text)

cat("After chopping the sentences, we have", nrow(tokenized_df), "individual words!\n")

In [ ]:
# =====================================================================
# STEP 5: Stopwords Removal (Throwing away useless words)
# =====================================================================
# "Stopwords" are words like "the", "and", "is", "at".
# They are required for human grammar, but they have zero emotion.
# If we ask the computer "Is the word 'the' happy or sad?", it gets confused.
# So, we use 'remove_stopwords' to delete them completely.

clean_tokens_df <- remove_stopwords(tokenized_df, word_column = "word")

cat("After throwing away useless stopwords, we only have", nrow(clean_tokens_df), "important words left.\n")

# The token file is used only to connect a review ID to each cleaned word.
# Keeping only these two columns avoids repeating full review text and source
# metadata thousands of times in the tracked CSV output.
clean_tokens_for_output <- clean_tokens_df %>%
  select(review_id, word)

In [ ]:
# =====================================================================
# STEP 6: Save the Clean Data
# =====================================================================
# We now have two beautiful, clean tables. 
# 1. cleaned_reviews_df: The full sentences (used for scoring the hotel).
# 2. clean_tokens_df: The chopped individual words (used for the wordcloud).

# Let's save them to the computer!
dir.create(dirname(cleaned_reviews_path), recursive = TRUE, showWarnings = FALSE)

write_csv(cleaned_reviews_df, cleaned_reviews_path)
write_csv(clean_tokens_for_output, cleaned_tokens_path)

cat("Cleaned reviews & tokens successfully saved to the 'data/cleaned/' folder!\n")

## Step 3: Sentiment Lexicon Scoring

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 3: Sentiment Lexicon Scoring
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# How does a computer know if a review is happy or angry? It uses a "Lexicon".
# A Lexicon is simply a giant dictionary where scientists have manually assigned 
# numeric scores to words. For example: "excellent" = +4, "dirty" = -3.
# We will scan our clean reviews against these dictionaries to calculate a final score.

In [ ]:
# =====================================================================
# STEP 1: Load Required Packages
# =====================================================================
# 'tidyverse' and 'tidytext' help us read and manipulate our tables.
library(tidyverse)
library(tidytext)

# 'syuzhet' is a very famous toolset built specifically for Sentiment Analysis.
# It contains the massive dictionaries we need to score emotions.
library(syuzhet)
source("scripts/data_config.R")
source("scripts/helpers.R")

cat("Libraries loaded successfully!\n")

In [ ]:
# =====================================================================
# STEP 2: Open the Cleaned Reviews
# =====================================================================
# We load the full cleaned sentences from Step 2 into a box named 'cleaned_reviews'.
if (!file.exists(cleaned_reviews_path)) {
  stop(
    paste(
      "Missing cleaned review data at:",
      cleaned_reviews_path,
      "Run Step 2 first with: Rscript 02_cleaning.R"
    ),
    call. = FALSE
  )
}

cleaned_reviews <- read_csv(cleaned_reviews_path, show_col_types = FALSE)

cat("Loaded", nrow(cleaned_reviews), "cleaned reviews for emotional scoring.\n")

In [ ]:
# =====================================================================
# STEP 3: Apply the Dictionary Lexicons (Syuzhet & AFINN)
# =====================================================================
# We are asking the computer to read every review and calculate a math score.
# We use two different methods to be thorough:
# 1. "syuzhet" method: A standard scientific scoring system.
# 2. "afinn" method: The word-level lexicon runs from -5 (furious) to
#    +5 (thrilled). For a full review, the method adds those word scores
#    together, so score_afinn is a summed review score and can be above +5
#    or below -5.
# The helper below flips sentiment words that appear right after negators such
# as "not", "never", "without", or "cannot". This keeps phrases like "cannot
# recommend" from being counted as positive just because "recommend" is positive.
# We also count the words in each cleaned review. The raw sentiment scores are
# still saved, but the workflow also creates length-normalized scores using the
# median review length in this dataset. That means the denominator comes from the
# hotel's own data instead of from a round number chosen by hand.

# 'mutate' means "create a new column in our table".
sentiment_results <- cleaned_reviews %>%
  mutate(
    # review_word_count stores how long each cleaned review is.
    # We need this because raw sentiment scores grow when a review has more words.
    review_word_count = count_cleaned_words(cleaned_text),
    score_syuzhet = score_negation_aware_sentiment(cleaned_text, method = "syuzhet"),
    score_afinn = score_negation_aware_sentiment(cleaned_text, method = "afinn")
  )

# Find the "typical" review length in this dataset.
# We ignore empty reviews because a review with 0 words cannot be used as a
# sensible denominator.
median_review_word_count <- sentiment_results %>%
  filter(review_word_count > 0) %>%
  summarise(value = median(review_word_count), .groups = "drop") %>%
  pull(value)

# If every cleaned review somehow had 0 words, normalization would be impossible.
# Stop here with a plain explanation instead of creating misleading NA scores.
if (length(median_review_word_count) == 0 || is.na(median_review_word_count)) {
  stop("No cleaned words were found, so sentiment scores cannot be length-normalized.", call. = FALSE)
}

sentiment_results <- sentiment_results %>%
  mutate(
    median_review_word_count = median_review_word_count,
    # These normalized scores keep the original direction of sentiment but put
    # short and long reviews onto the same typical-review length.
    score_syuzhet_per_median_review = normalize_score_to_word_count(
      score_syuzhet,
      review_word_count,
      median_review_word_count
    ),
    score_afinn_per_median_review = normalize_score_to_word_count(
      score_afinn,
      review_word_count,
      median_review_word_count
    )
  )

cat("Median cleaned review length used for normalization:", median_review_word_count, "words\n")

In [ ]:
# =====================================================================
# STEP 4: Classify the Scores into Simple Categories
# =====================================================================
# A numeric score like "1.45" is hard to understand.
# We use 'case_when' (which is like a giant IF statement) to categorize them:
# IF the score is greater than 0  -> "Positive"
# IF the score is less than 0     -> "Negative"
# IF the score is exactly 0       -> "Neutral"

sentiment_classified <- sentiment_results %>%
  mutate(
    sentiment_category = case_when(
      score_syuzhet > 0 ~ "Positive",
      score_syuzhet < 0 ~ "Negative",
      TRUE              ~ "Neutral" # TRUE here means "everything else"
    )
  )

# Now, let's group all the "Positive" and "Negative" rows together and count them!
sentiment_summary <- sentiment_classified %>%
  group_by(sentiment_category) %>%
  summarise(
    count = n(), # n() counts the number of rows
    percentage = (n() / nrow(sentiment_classified)) * 100 # Calculate the percentage %
  )

# Print the final report to the screen
print(sentiment_summary)

In [ ]:
# =====================================================================
# STEP 5: NRC Deep Emotion Extraction
# =====================================================================
# "Positive" or "Negative" is too basic. What if we want to know if guests are ANGRY or AFRAID?
# The 'NRC' lexicon checks text against 8 human emotions:
# (anger, anticipation, disgust, fear, joy, sadness, surprise, trust).
#
# UNDERSTANDING NRC SCORES: The output scores for NRC emotions represent the frequency 
# or counts of words associated with each specific emotion category found in the text. 
# For example, a "joy" score of 3 means there were 3 words in the review associated with joy.

cat("Processing deep emotions. The computer is reading thousands of words, please wait a few seconds...\n")

# 'get_nrc_sentiment' returns a table showing exactly how much 'joy' or 'anger' is in the text.
nrc_emotions <- get_nrc_sentiment(sentiment_classified$cleaned_text)

# 'bind_cols' acts like glue. It takes our original table, and glues the new emotion columns to the right side of it.
final_research_data <- bind_cols(sentiment_classified, nrc_emotions)

In [ ]:
# =====================================================================
# STEP 6: Save the Scored Data
# =====================================================================
# Our table now has the original text, the numeric scores, the positive/negative labels, AND the 8 deep emotions!
# We save this final master dataset to the computer.

write_csv(final_research_data, sentiment_scores_path)
cat("Sentiment-scored research data saved successfully!\n")

## Step 4: Data Visualization and Insights

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 4: Data Visualization & Insights
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# Math numbers and huge spreadsheets are boring and hard to read.
# "Data Visualization" means turning those numbers into beautiful, colorful 
# pictures (like bar charts and line graphs) so we can easily understand the story!

In [ ]:
# =====================================================================
# STEP 1: Load Packages and Data
# =====================================================================
# We load our toolsets again. 
library(tidyverse)    # Contains 'ggplot2' - the most powerful charting tool in R.
library(tidytext)     # For handling our text data.
library(wordcloud)    # A fun tool specifically for drawing Word Clouds.
library(RColorBrewer) # A tool that provides beautiful, professional color palettes.
source("scripts/data_config.R")
source("scripts/helpers.R")

# We load the final, scored dataset from Step 3.
if (!file.exists(sentiment_scores_path) || !file.exists(cleaned_tokens_path)) {
  stop(
    paste(
      "Missing cleaned analysis data.",
      "Run Step 2 and Step 3 before visualization."
    ),
    call. = FALSE
  )
}

research_data <- read_csv(sentiment_scores_path, show_col_types = FALSE)
# We also load the chopped up individual words from Step 2 (for the word cloud).
cleaned_tokens <- read_csv(cleaned_tokens_path, show_col_types = FALSE)

dir.create(figures_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(reports_dir, recursive = TRUE, showWarnings = FALSE)

cat("Data and drawing tools loaded successfully!\n")

In [ ]:
# =====================================================================
# STEP 2: Create a Custom "Theme" (Paintbrush)
# =====================================================================
# Instead of telling the computer "make the title bold and size 16" every 
# single time we draw a chart, we create a 'theme_premium' recipe once.
# Now, we just slap 'theme_premium()' on any chart, and it instantly looks professional!

theme_premium <- function() {
  theme_minimal() + # Start with a clean, white background
  theme(
    text = element_text(color = "#2C3E50"), # Dark blue/grey text
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5, color = "#1A252C"), # Centered bold title
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "#5A6F7C"), # Centered subtitle
    axis.title = element_text(face = "bold", size = 11), # Bold axis labels (X and Y)
    legend.position = "none", # Hide the messy legend box
    panel.grid.minor = element_blank() # Remove distracting background grid lines
  )
}

# RStudio has an interactive Plots pane, but Rscript runs without one.
# This helper only prints charts when a person is using R interactively.
# The charts are still saved as PNG files by ggsave() in every run.
show_plot_for_interactive_use <- function(plot_object) {
  if (interactive()) {
    print(plot_object)
  }
}

# When a dataset is too small for a specific chart, we still save a PNG file.
# This keeps the output folder honest: readers see why a chart is empty instead
# of accidentally reading an older chart from a previous run.
save_placeholder_plot <- function(file_name, title, message, width = 10, height = 6) {
  placeholder_plot <- ggplot() +
    annotate(
      "text",
      x = 0,
      y = 0,
      label = message,
      size = 4,
      color = "#2C3E50",
      lineheight = 0.95
    ) +
    labs(title = title, x = NULL, y = NULL) +
    theme_void() +
    theme(
      plot.title = element_text(face = "bold", size = 15, hjust = 0.5, color = "#1A252C")
    )

  ggsave(file.path(figures_dir, file_name), plot = placeholder_plot, width = width, height = height, dpi = 300)
  show_plot_for_interactive_use(placeholder_plot)
}

parse_review_dates <- function(date_values) {
  # Dates can be written in different formats.
  # This helper only accepts dates that include a year. A label such as "May 18"
  # is ambiguous because 18 could be a day, but lubridate::my() can treat it as
  # the year 2018. We return NA for that case so the workflow can stop with a
  # clear message instead of drawing the review in the wrong year.
  date_text <- str_squish(as.character(date_values))
  date_text[date_text %in% c("", "NA", "N/A", "NULL")] <- NA_character_
  parsed <- as.Date(rep(NA_character_, length(date_text)))

  parse_matching_dates <- function(pattern, parser) {
    matching_dates <- is.na(parsed) &
      !is.na(date_text) &
      str_detect(date_text, regex(pattern, ignore_case = TRUE))

    if (any(matching_dates)) {
      parsed[matching_dates] <<- as.Date(suppressWarnings(parser(date_text[matching_dates])))
    }
  }

  parse_matching_dates("^\\d{4}[-/]\\d{1,2}[-/]\\d{1,2}$", lubridate::ymd)
  parse_matching_dates("^\\d{4}[-/]\\d{1,2}$", lubridate::ym)
  parse_matching_dates("^[A-Za-z]{3,9}\\s+\\d{1,2},\\s*\\d{4}$", lubridate::mdy)
  parse_matching_dates("^\\d{1,2}\\s+[A-Za-z]{3,9}\\s+\\d{4}$", lubridate::dmy)
  parse_matching_dates("^[A-Za-z]{3,9}\\s+\\d{4}$", lubridate::my)

  parsed
}

safe_correlation <- function(x_values, y_values) {
  complete_rows <- complete.cases(x_values, y_values)
  if (sum(complete_rows) < 3) {
    return(NA_real_)
  }

  x_complete <- x_values[complete_rows]
  y_complete <- y_values[complete_rows]
  if (sd(x_complete) == 0 || sd(y_complete) == 0) {
    return(NA_real_)
  }

  cor(x_complete, y_complete)
}

remove_existing_files <- function(paths) {
  existing_paths <- paths[file.exists(paths)]
  if (length(existing_paths) > 0) {
    invisible(file.remove(existing_paths))
  }
}

aspect_report_paths <- file.path(
  reports_dir,
  c(
    "aspect_rating_summary.csv",
    "high_overall_low_aspect_reviews.csv",
    "aspect_text_alignment.csv",
    "aspect_text_band_summary.csv",
    "aspect_text_key_terms.csv",
    "aspect_text_key_phrases.csv",
    "aspect_text_mismatches.csv",
    "aspect_qualitative_examples.csv"
  )
)

aspect_figure_paths <- file.path(
  figures_dir,
  c(
    "aspect_mean_ratings.png",
    "aspect_low_score_share.png",
    "aspect_yearly_rating_heatmap.png",
    "aspect_sentiment_by_rating_boxplot.png",
    "aspect_low_score_key_terms.png",
    "aspect_low_score_key_phrases.png",
    "aspect_negative_term_heatmap.png",
    "aspect_text_mismatch_counts.png"
  )
)

clear_aspect_outputs <- function() {
  remove_existing_files(c(aspect_report_paths, aspect_figure_paths))
}

In [ ]:
# =====================================================================
# STEP 3: Draw the Sentiment Distribution Chart (Positive vs Negative)
# =====================================================================
# We count how many reviews fall into each sentiment category.
sentiment_counts <- research_data %>%
  count(sentiment_category, name = "review_count")

# We use 'ggplot' to draw the chart.
# Think of 'aes' (aesthetics) as telling the computer where to put the data.
# x = sentiment_category means "Put Positive, Neutral, and Negative on the bottom X axis".
# y = review_count means "Make each bar as tall as the number of reviews in that category".
# fill = sentiment_category means "Color them based on their category".

p1 <- ggplot(sentiment_counts, aes(x = sentiment_category, y = review_count, fill = sentiment_category)) +
  geom_col(width = 0.6, alpha = 0.85) + # Draw solid bars (alpha makes them slightly transparent)
  geom_text(aes(label = review_count), vjust = -0.4, fontface = "bold", color = "#2C3E50") +
  # scale_fill_manual lets us pick the exact paint colors. Green = Good, Red = Bad, Grey = Neutral.
  scale_fill_manual(values = c("Positive" = "#16A085", "Negative" = "#E74C3C", "Neutral" = "#95A5A6")) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.12))) +
  # 'labs' stands for labels. We give our chart a title.
  labs(
    title = "Sentiment Distribution of TripAdvisor Reviews",
    subtitle = "Bvlgari Resort Bali guest opinions from the prepared TripAdvisor review dataset",
    x = "Sentiment Category",
    y = "Number of Reviews"
  ) +
  theme_premium() # Apply our custom paintbrush from Step 2!

# Show the chart in RStudio, but do not open a PDF device during Rscript runs.
show_plot_for_interactive_use(p1)

# 'ggsave' saves the picture to our computer folder as a PNG image file.
ggsave(file.path(figures_dir, "sentiment_distribution.png"), plot = p1, width = 7, height = 5, dpi = 300)

# ---------------------------------------------------------------------
# Extra Chart: Compare Star Ratings with Text Sentiment
# ---------------------------------------------------------------------
# A guest gives the hotel a star rating (1 to 5), but they also write a review.
# These two things are related, but they are not exactly the same.
# For example, a guest might give 5 stars overall but still complain about one
# small detail in the written review.
#
# This chart groups the reviews by their TripAdvisor star rating and then
# shows the spread of the AFINN text sentiment scores inside each rating group.
# If the boxes move upward from 1 star to 5 stars, it means the text sentiment
# generally agrees with the star ratings.
rating_sentiment <- research_data %>%
  mutate(
    # The rating column should already be numeric, but parse_number() makes this
    # safer if the data ever contains text like "5 of 5" instead of just "5".
    rating_number = readr::parse_number(as.character(rating)),
    rating_group = factor(
      rating_number,
      levels = 1:5,
      labels = paste(1:5, "star")
    )
  ) %>%
  filter(!is.na(rating_group), !is.na(score_afinn))

# Count how many reviews are inside each rating group.
# We print these counts below the boxes because a box based on 596 reviews is
# much stronger evidence than a box based on only 22 reviews.
rating_counts <- rating_sentiment %>%
  count(rating_number, rating_group, name = "review_count")

# Calculate the average TripAdvisor star rating across all reviews.
# We draw this as a dashed vertical line, so students can see where the overall
# hotel rating sits on the 1-to-5 rating scale.
average_rating <- mean(rating_sentiment$rating_number)

# These numbers create a little empty space below the boxes.
# That space is where we place the "n=" labels without covering the chart.
rating_score_min <- min(rating_sentiment$score_afinn)
rating_score_max <- max(rating_sentiment$score_afinn)
rating_score_range <- max(rating_score_max - rating_score_min, 1)
rating_label_y <- rating_score_min - rating_score_range * 0.08
rating_axis_floor <- rating_label_y - rating_score_range * 0.05
rating_axis_ceiling <- rating_score_max + rating_score_range * 0.12
rating_average_line <- tibble(
  x = average_rating,
  y = rating_axis_floor,
  yend = rating_axis_ceiling
)

p1b <- ggplot(rating_sentiment, aes(x = rating_number, y = score_afinn, group = rating_number, fill = rating_group)) +
  # The boxplot shows the sentiment distribution for each rating.
  # The thick line inside each box is the median sentiment score.
  # Dots outside the whiskers are unusual reviews compared with the rest of
  # that same rating group.
  geom_boxplot(alpha = 0.82, outlier.alpha = 0.35, width = 0.62) +
  # This dashed line marks the average star rating for the full dataset.
  geom_segment(
    data = rating_average_line,
    aes(x = x, xend = x, y = y, yend = yend),
    inherit.aes = FALSE,
    linetype = "dashed",
    color = "#2C3E50",
    linewidth = 0.8
  ) +
  annotate(
    "label",
    x = average_rating,
    y = rating_axis_ceiling,
    label = paste0("Average rating: ", round(average_rating, 2), "/5"),
    vjust = 1.1,
    size = 3,
    color = "#2C3E50",
    fill = "white"
  ) +
  # A diamond marker is added for the average sentiment score in each group.
  # This is different from the median line inside the box.
  stat_summary(
    fun = mean,
    geom = "point",
    shape = 23,
    size = 3,
    fill = "#2C3E50",
    color = "white"
  ) +
  geom_text(
    data = rating_counts,
    aes(x = rating_number, y = rating_label_y, label = paste0("n=", review_count)),
    inherit.aes = FALSE,
    fontface = "bold",
    color = "#2C3E50",
    size = 3.2
  ) +
  scale_fill_brewer(palette = "RdYlGn") +
  scale_x_continuous(breaks = 1:5, labels = paste(1:5, "star"), expand = expansion(mult = c(0.08, 0.08))) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.02))) +
  coord_cartesian(ylim = c(rating_axis_floor, rating_axis_ceiling)) +
  labs(
    title = "Sentiment Score Distribution by TripAdvisor Rating",
    subtitle = "Boxes show text sentiment by star rating; diamonds show group sentiment averages\nDashed line shows the overall average TripAdvisor rating",
    x = "TripAdvisor Rating",
    y = "AFINN Sentiment Score"
  ) +
  theme_premium()

show_plot_for_interactive_use(p1b)

ggsave(file.path(figures_dir, "sentiment_by_rating_boxplot.png"), plot = p1b, width = 9.5, height = 5.8, dpi = 300)

# ---------------------------------------------------------------------
# Extra Table: Annual Review Profile
# ---------------------------------------------------------------------
# The reference article includes an annual profile table with three ideas:
# time, reviewer geography, and rating distribution. This project can fully
# summarize time and ratings. When reviewer_location is present, it is used only
# as an aggregate region field, not as a person-level identifier.
minimum_reviews_for_region_listing <- 2
maximum_regions_listed_per_year <- 4
possible_geography_columns <- c(
  "reviewer_location",
  "reviewer_region",
  "reviewer_country",
  "guest_region",
  "guest_country",
  "origin_region",
  "origin_country",
  "region",
  "country"
)
available_geography_columns <- possible_geography_columns[possible_geography_columns %in% names(research_data)]
if (length(available_geography_columns) > 0) {
  available_geography_column <- available_geography_columns[[1]]
  geography_source_label <- paste("Source column:", available_geography_column)
} else {
  available_geography_column <- NA_character_
  geography_source_label <- "No non-identifying reviewer geography field in prepared dataset"
}

# Start with one clean row per review for the annual profile table.
# We parse dates and ratings as numbers so R can group reviews by year and count
# how many 5-star, 4-star, 3-star, 2-star, and 1-star reviews appear each year.
annual_profile_reviews <- research_data %>%
  mutate(
    date_parsed = parse_review_dates(review_date),
    year = as.character(lubridate::year(date_parsed)),
    rating_number = as.integer(round(readr::parse_number(as.character(rating))))
  ) %>%
  filter(!is.na(date_parsed), !is.na(year), !is.na(rating_number), rating_number %in% 1:5)

# This small helper counts the number of reviews at each star level.
# The final columns are ordered from 5 stars down to 1 star to match the paper
# style in the reference image.
build_rating_distribution <- function(data) {
  rating_columns <- paste0("rating_", 5:1, "_star_reviews")

  data %>%
    count(year, rating_number, name = "rating_count") %>%
    tidyr::complete(year, rating_number = 1:5, fill = list(rating_count = 0)) %>%
    mutate(rating_column = paste0("rating_", rating_number, "_star_reviews")) %>%
    select(year, rating_column, rating_count) %>%
    pivot_wider(names_from = rating_column, values_from = rating_count, values_fill = 0) %>%
    select(year, all_of(rating_columns))
}

# This helper lists the most common reviewer locations that appear at least
# twice in the same year. Missing locations are labeled "Unknown" so the table
# is honest about incomplete geography coverage. The list is capped because a
# long free-text location column can otherwise make the report unreadable.
build_region_summary <- function(data) {
  if (is.na(available_geography_column)) {
    return(
      data %>%
        distinct(year) %>%
        mutate(regions_with_minimum_reviews = "Not available in prepared dataset")
    )
  }

  region_counts <- data %>%
    mutate(region_name = str_squish(as.character(.data[[available_geography_column]]))) %>%
    mutate(
      region_name = na_if(region_name, ""),
      region_name = replace_na(region_name, "Unknown")
    ) %>%
    count(year, region_name, name = "review_count") %>%
    filter(review_count >= minimum_reviews_for_region_listing) %>%
    arrange(year, desc(review_count), region_name)

  all_years <- data %>%
    distinct(year)

  if (nrow(region_counts) == 0) {
    return(
      all_years %>%
        mutate(regions_with_minimum_reviews = "No region reached the minimum review count")
    )
  }

  region_counts %>%
    group_by(year) %>%
    mutate(
      total_regions_at_threshold = n(),
      region_rank = row_number()
    ) %>%
    filter(region_rank <= maximum_regions_listed_per_year) %>%
    summarise(
      listed_regions = paste0(region_name, " (", review_count, ")", collapse = ", "),
      omitted_region_count = max(total_regions_at_threshold) - n(),
      .groups = "drop"
    ) %>%
    mutate(
      regions_with_minimum_reviews = if_else(
        omitted_region_count > 0,
        paste0(listed_regions, ", plus ", omitted_region_count, " other locations"),
        listed_regions
      )
    ) %>%
    select(year, regions_with_minimum_reviews) %>%
    right_join(all_years, by = "year") %>%
    mutate(
      regions_with_minimum_reviews = replace_na(
        regions_with_minimum_reviews,
        "No region reached the minimum review count"
      )
    )
}

annual_rating_distribution <- build_rating_distribution(annual_profile_reviews)
annual_region_summary <- build_region_summary(annual_profile_reviews)

# Combine the yearly review counts, date range, geography status, rating
# average, and rating distribution into one CSV table for the paper.
annual_review_profile_years <- annual_profile_reviews %>%
  group_by(year) %>%
  summarise(
    review_count = n(),
    review_date_start = min(date_parsed),
    review_date_end = max(date_parsed),
    rating_average = round(mean(rating_number), 2),
    .groups = "drop"
  ) %>%
  left_join(annual_region_summary, by = "year") %>%
  left_join(annual_rating_distribution, by = "year") %>%
  mutate(geography_source = geography_source_label) %>%
  arrange(desc(as.integer(year)))

annual_review_profile_total <- annual_profile_reviews %>%
  mutate(year = "Total") %>%
  group_by(year) %>%
  summarise(
    review_count = n(),
    review_date_start = min(date_parsed),
    review_date_end = max(date_parsed),
    rating_average = round(mean(rating_number), 2),
    .groups = "drop"
  ) %>%
  left_join(build_region_summary(mutate(annual_profile_reviews, year = "Total")), by = "year") %>%
  left_join(build_rating_distribution(mutate(annual_profile_reviews, year = "Total")), by = "year") %>%
  mutate(geography_source = geography_source_label)

annual_review_profile <- bind_rows(
  annual_review_profile_years,
  annual_review_profile_total
) %>%
  select(
    year,
    review_count,
    review_date_start,
    review_date_end,
    regions_with_minimum_reviews,
    geography_source,
    rating_average,
    rating_5_star_reviews,
    rating_4_star_reviews,
    rating_3_star_reviews,
    rating_2_star_reviews,
    rating_1_star_reviews
  )

write_csv(annual_review_profile, annual_review_profile_path)

In [ ]:
# =====================================================================
# STEP 4: Analyze Structured Aspect Ratings
# =====================================================================
# Many TripAdvisor reviews include separate ratings for value, rooms, location,
# cleanliness, service, and sleep quality. These fields are very useful because
# they tell us which part of the guest experience is weaker, even when the
# overall star rating is high.

aspect_rating_columns <- c(
  "value_rating" = "Value",
  "rooms_rating" = "Rooms",
  "location_rating" = "Location",
  "cleanliness_rating" = "Cleanliness",
  "service_rating" = "Service",
  "sleep_quality_rating" = "Sleep quality"
)
# The raw data may not always include every optional TripAdvisor aspect field.
# This line checks which of the expected aspect columns are actually available
# before the script tries to use them.
available_aspect_columns <- names(aspect_rating_columns)[names(aspect_rating_columns) %in% names(research_data)]
has_usable_aspect_ratings <- FALSE

if (length(available_aspect_columns) > 0) {
  # Make sure the rating columns are numbers. parse_number() is forgiving: it
  # can read values such as "5", "5.0", or even "5 of 5" as the number 5.
  aspect_data <- research_data %>%
    prepare_length_normalized_sentiment() %>%
    mutate(
      overall_rating = readr::parse_number(as.character(rating)),
      # Raw AFINN is a summed score. Longer reviews can therefore produce larger
      # scores even when the emotional density is similar. The aspect summaries
      # use scores scaled to the median review length in this dataset so low/high
      # aspect comparisons are fairer without choosing an arbitrary denominator.
      score_afinn_number = score_afinn_per_median_review,
      date_parsed = parse_review_dates(review_date),
      year_label = as.character(lubridate::year(date_parsed)),
      across(all_of(available_aspect_columns), ~ readr::parse_number(as.character(.x)))
    )

  usable_aspect_columns <- available_aspect_columns[
    vapply(aspect_data[available_aspect_columns], function(column) any(!is.na(column)), logical(1))
  ]

  if (length(usable_aspect_columns) == 0) {
    clear_aspect_outputs()
    message("No usable structured aspect ratings were found, so Step 4 is skipping aspect reports and charts.")
  } else {
    has_usable_aspect_ratings <- TRUE
    available_aspect_columns <- usable_aspect_columns

    # pivot_longer changes the table shape from "wide" to "long":
    # - Wide format has one row per review and many aspect-rating columns.
    # - Long format has one row per review-aspect pair.
    # Long format makes summaries and charts much easier to build.
    aspect_long <- aspect_data %>%
      select(
        review_id,
        review_date,
        year_label,
        title,
        review_text,
        overall_rating,
        score_afinn_raw,
        review_word_count,
        median_review_word_count,
        score_afinn_per_median_review,
        score_afinn_number,
        all_of(available_aspect_columns)
      ) %>%
      pivot_longer(
        cols = all_of(available_aspect_columns),
        names_to = "aspect_key",
        values_to = "aspect_rating"
      ) %>%
      mutate(
        aspect_label = factor(
          aspect_rating_columns[aspect_key],
          levels = unname(aspect_rating_columns)
        )
      )

  # This table gives one summary row for each aspect. It includes:
  # - coverage: how many reviews supplied that optional rating
  # - mean and median: typical score
  # - low-score share: how often guests gave that aspect a 1, 2, or 3
  # - correlations: whether the aspect tends to move with overall rating or text sentiment
  aspect_summary <- aspect_long %>%
    group_by(aspect_key, aspect_label) %>%
    summarise(
      reviews_with_rating = sum(!is.na(aspect_rating)),
      coverage_pct = mean(!is.na(aspect_rating)),
      mean_rating = mean(aspect_rating, na.rm = TRUE),
      median_rating = median(aspect_rating, na.rm = TRUE),
      low_score_count = sum(aspect_rating <= 3, na.rm = TRUE),
      low_score_share = if_else(reviews_with_rating > 0, low_score_count / reviews_with_rating, NA_real_),
      very_low_score_count = sum(aspect_rating <= 2, na.rm = TRUE),
      very_low_score_share = if_else(reviews_with_rating > 0, very_low_score_count / reviews_with_rating, NA_real_),
      corr_overall_rating = safe_correlation(aspect_rating, overall_rating),
      corr_afinn = safe_correlation(aspect_rating, score_afinn_number),
      .groups = "drop"
    ) %>%
    arrange(mean_rating)

  # Save a CSV report so the exact numbers behind the charts are easy to inspect
  # or reuse in the analysis report.
  write_csv(
    aspect_summary %>%
      transmute(
        aspect_key,
        aspect_label,
        reviews_with_rating,
        coverage_pct = round(coverage_pct * 100, 1),
        mean_rating = round(mean_rating, 2),
        median_rating = round(median_rating, 2),
        low_score_count,
        low_score_pct = round(low_score_share * 100, 1),
        very_low_score_count,
        very_low_score_pct = round(very_low_score_share * 100, 1),
        corr_overall_rating = round(corr_overall_rating, 2),
        corr_afinn = round(corr_afinn, 2)
      ),
    file.path(reports_dir, "aspect_rating_summary.csv")
  )

  # These rows are important because they show satisfied guests who still
  # reported a specific problem. Example: a 5-star review with Value = 2.
  high_overall_low_aspect_reviews <- aspect_long %>%
    filter(
      !is.na(overall_rating),
      overall_rating >= 4,
      !is.na(aspect_rating),
      aspect_rating <= 3
    ) %>%
    arrange(review_id, aspect_label) %>%
    group_by(review_id) %>%
    summarise(
      review_date = first(review_date),
      rating = first(overall_rating),
      low_aspects = paste(paste0(aspect_label, "=", aspect_rating), collapse = "; "),
      score_afinn_per_median_review = first(score_afinn_number),
      score_afinn_raw = first(score_afinn_raw),
      title = first(title),
      review_text = first(review_text),
      .groups = "drop"
    )

  # Save those special cases for manual reading.
  write_csv(
    high_overall_low_aspect_reviews,
    file.path(reports_dir, "high_overall_low_aspect_reviews.csv")
  )

  # Use consistent colors for the same aspects across every aspect chart.
  aspect_colors <- c(
    "Value" = "#E76F51",
    "Rooms" = "#457B9D",
    "Location" = "#2A9D8F",
    "Cleanliness" = "#6D597A",
    "Service" = "#264653",
    "Sleep quality" = "#F4A261"
  )

  # Chart 1: show which aspect has the highest or lowest average rating.
  p_aspect_mean <- ggplot(
    aspect_summary,
    aes(x = reorder(aspect_label, mean_rating), y = mean_rating, fill = aspect_label)
  ) +
    geom_col(width = 0.68, alpha = 0.88) +
    geom_text(
      aes(label = paste0(round(mean_rating, 2), " / 5\nn=", reviews_with_rating)),
      hjust = -0.08,
      fontface = "bold",
      color = "#2C3E50",
      size = 3
    ) +
    coord_flip() +
    scale_fill_manual(values = aspect_colors) +
    scale_y_continuous(limits = c(0, 5), expand = expansion(mult = c(0, 0.18))) +
    labs(
      title = "Average Guest Experience Aspect Ratings",
      subtitle = "Structured TripAdvisor aspect scores show which experience dimensions are strongest or weakest",
      x = "Aspect",
      y = "Average rating"
    ) +
    theme_premium()

  show_plot_for_interactive_use(p_aspect_mean)

  ggsave(file.path(figures_dir, "aspect_mean_ratings.png"), plot = p_aspect_mean, width = 8.5, height = 5.4, dpi = 300)

  # Chart 2: show where low scores are concentrated. This can reveal a problem
  # even if the average rating is still high.
  p_aspect_low <- ggplot(
    aspect_summary,
    aes(x = reorder(aspect_label, low_score_share), y = low_score_share, fill = aspect_label)
  ) +
    geom_col(width = 0.68, alpha = 0.88) +
    geom_text(
      aes(label = paste0(scales::percent(low_score_share, accuracy = 0.1), "\nn=", low_score_count)),
      hjust = -0.08,
      fontface = "bold",
      color = "#2C3E50",
      size = 3
    ) +
    coord_flip() +
    scale_fill_manual(values = aspect_colors) +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1), expand = expansion(mult = c(0, 0.18))) +
    labs(
      title = "Low-Score Share by Guest Experience Aspect",
      subtitle = "Low-score share counts aspect ratings from 1 to 3 stars",
      x = "Aspect",
      y = "Share of aspect ratings at 1-3 stars"
    ) +
    theme_premium()

  show_plot_for_interactive_use(p_aspect_low)

  ggsave(file.path(figures_dir, "aspect_low_score_share.png"), plot = p_aspect_low, width = 8.5, height = 5.4, dpi = 300)

  # Chart 3: summarize aspect ratings by year. Years with very few ratings are
  # colored grey because tiny sample sizes should not be over-interpreted.
  aspect_yearly <- aspect_long %>%
    filter(!is.na(year_label), !is.na(aspect_rating)) %>%
    group_by(year_label, aspect_label) %>%
    summarise(
      mean_rating = mean(aspect_rating),
      review_count = n(),
      .groups = "drop"
    ) %>%
    mutate(
      year_label = factor(year_label, levels = sort(unique(year_label))),
      rating_for_fill = if_else(review_count >= 5, mean_rating, NA_real_),
      heatmap_label = if_else(
        review_count >= 5,
        paste0(round(mean_rating, 1), "\nn=", review_count),
        paste0("n=", review_count)
      )
    )

  p_aspect_heatmap <- ggplot(aspect_yearly, aes(x = aspect_label, y = year_label, fill = rating_for_fill)) +
    geom_tile(color = "white", linewidth = 0.4) +
    geom_text(aes(label = heatmap_label), size = 2.3, color = "#2C3E50", lineheight = 0.85) +
    scale_fill_gradient2(
      low = "#E76F51",
      mid = "#F7F9F9",
      high = "#2A9D8F",
      midpoint = 4,
      limits = c(1, 5),
      na.value = "#ECEFF1",
      name = "Average\nrating"
    ) +
    labs(
      title = "Yearly Aspect Rating Heatmap",
      subtitle = "Grey tiles have fewer than 5 aspect ratings; labels show average rating and n when enough data exists",
      x = "Aspect",
      y = "Year"
    ) +
    theme_premium() +
    theme(
      legend.position = "right",
      axis.text.x = element_text(angle = 35, hjust = 1)
    )

  show_plot_for_interactive_use(p_aspect_heatmap)

  ggsave(file.path(figures_dir, "aspect_yearly_rating_heatmap.png"), plot = p_aspect_heatmap, width = 10, height = 7, dpi = 300)

    cat("Aspect rating analysis complete!\n")
    cat("- ", file.path(reports_dir, "aspect_rating_summary.csv"), "\n", sep = "")
    cat("- ", file.path(reports_dir, "high_overall_low_aspect_reviews.csv"), "\n", sep = "")
  }
} else {
  clear_aspect_outputs()
  warning("No structured aspect rating columns were found in the scored dataset.")
}

In [ ]:
# =====================================================================
# STEP 5: Draw the Deep Emotion Chart (Joy, Trust, Anger, etc.)
# =====================================================================
# We calculate the total sum of all the 8 emotions.
emotions_summary <- research_data %>%
  select(anger, anticipation, disgust, fear, joy, sadness, surprise, trust) %>%
  summarise(across(everything(), sum)) %>% # Add them all up
  pivot_longer(cols = everything(), names_to = "emotion", values_to = "score") %>% # Reshape the table so we can graph it
  arrange(desc(score)) # Sort them from highest to lowest

# Draw the chart!
# 'reorder' automatically sorts the bars so the tallest one is at the top.
p2 <- ggplot(emotions_summary, aes(x = reorder(emotion, score), y = score, fill = emotion)) +
  geom_col(alpha = 0.85, width = 0.7) + # Draw the columns
  geom_text(aes(label = format(score, big.mark = ",")), hjust = -0.15, fontface = "bold", color = "#2C3E50") +
  coord_flip() + # Flip the chart sideways! Horizontal charts are easier to read.
  scale_fill_brewer(palette = "Dark2") + # Use a built-in professional color palette
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(
    title = "Detailed Emotional Profile of Guest Experience",
    subtitle = "NRC Lexicon emotional analysis metrics for hotel reviews",
    x = "Emotion Dimension",
    y = "Total Emotion Score"
  ) +
  theme_premium()

# Show the chart in RStudio, but do not open a PDF device during Rscript runs.
show_plot_for_interactive_use(p2)

ggsave(file.path(figures_dir, "emotions_breakdown.png"), plot = p2, width = 8, height = 5, dpi = 300)

In [ ]:
# =====================================================================
# STEP 6: Draw the Word Cloud
# =====================================================================
# This word cloud is sentiment-only.
# That means we do NOT use every frequent word in the reviews.
# Instead, we first ask the AFINN sentiment dictionary which words have a
# positive or negative sentiment score. Words with score 0 are neutral, so they
# are removed from this chart.
# Some words can be sentiment words in one context but neutral hotel words in
# another context. For example, AFINN treats "lobby" as negative because of the
# political meaning, but a hotel lobby is just a place. We exclude those obvious
# hotel-context false positives here.
domain_neutral_words <- c("hotel", "resort", "tripadvisor", "review", "stay", "room", "villa", "service", "lobby")

sentiment_lexicon_words <- cleaned_tokens %>%
  distinct(word) %>%
  mutate(sentiment_score = syuzhet::get_sentiment(word, method = "afinn")) %>%
  filter(sentiment_score != 0) %>%
  filter(!word %in% domain_neutral_words)

# Now count only the review words that AFINN says are sentiment words.
# The bigger the word, the more often guests used that sentiment word.
sentiment_word_counts <- cleaned_tokens %>%
  inner_join(sentiment_lexicon_words, by = "word") %>%
  count(word, sentiment_score, sort = TRUE)

if (nrow(sentiment_word_counts) == 0) {
  message("No AFINN sentiment words were found in the cleaned review tokens, so Step 4 is saving a placeholder word cloud and continuing.")
  save_placeholder_plot(
    "wordcloud.png",
    "Sentiment Word Cloud",
    "No AFINN sentiment words were found after filtering the cleaned review tokens.",
    width = 5.4,
    height = 5.4
  )
} else {
  max_sentiment_cloud_words <- min(40, nrow(sentiment_word_counts))

  # Green words are positive sentiment words. Red words are negative sentiment words.
  sentiment_word_colors <- if_else(sentiment_word_counts$sentiment_score > 0, "#16A085", "#E74C3C")

  cat("Drawing the sentiment-only word cloud...\n")

  # Display the word cloud in the RStudio Plots tab only during interactive use.
  # In Rscript, the PNG file below is the intended saved output.
  if (interactive()) {
    set.seed(123)
    wordcloud(
      words = sentiment_word_counts$word,
      freq = sentiment_word_counts$n,
      min.freq = 2,
      max.words = max_sentiment_cloud_words,
      random.order = FALSE,
      rot.per = 0.2,
      ordered.colors = TRUE,
      colors = sentiment_word_colors
    )
  }

  # Open a digital "canvas" to draw the picture on
  png(file.path(figures_dir, "wordcloud.png"), width = 800, height = 800, res = 150)
  set.seed(123) # Lock the randomness so the cloud looks exactly the same every time we run it

  # Call the wordcloud drawing tool
  wordcloud(
    words = sentiment_word_counts$word, # What sentiment words to draw
    freq = sentiment_word_counts$n,     # How big to draw them based on frequency
    min.freq = 2,               # Ignore words that only appeared once
    max.words = max_sentiment_cloud_words, # Stop drawing after 40 words so it doesn't look messy
    random.order = FALSE,       # Put the biggest words right in the center
    rot.per = 0.2,              # Randomly rotate 20% of the words sideways
    ordered.colors = TRUE,      # Use each word's own positive/negative color
    colors = sentiment_word_colors # Green means positive, red means negative
  )
  dev.off() # Close the digital canvas and save the file!
}

In [ ]:
# =====================================================================
# STEP 7: Draw Sentiment Trend Charts
# =====================================================================
# We want to see if the hotel's review sentiment changes over time.
# This section creates several different time-based charts because each chart
# answers a slightly different research question:
# 1. Monthly heatmap: Which months/years look stronger or weaker?
# 2. Monthly rolling trend: Is sentiment moving up or down over time?
# 3. Quarterly boxplot: How much do reviews vary inside each quarter?
# 4. Yearly boxplot: How much do reviews vary inside each year?
trend_reviews <- research_data %>%
  prepare_length_normalized_sentiment() %>%
  mutate(
    # The TripAdvisor export may use full dates or month-year labels.
    date_parsed = parse_review_dates(review_date),
    # score_afinn is a raw summed word score. For trend charts, normalize it
    # by review length so long reviews do not dominate month and year averages.
    # The target length is the median cleaned review length for this dataset.
    score_afinn = score_afinn_per_median_review,
    rating_number = readr::parse_number(as.character(rating))
  )

unparseable_review_dates <- trend_reviews %>%
  filter(is.na(date_parsed), !is.na(review_date), str_squish(as.character(review_date)) != "") %>%
  distinct(review_date)

if (nrow(unparseable_review_dates) > 0) {
  stop(
    paste(
      "The review_date column contains unsupported or ambiguous month-day review dates.",
      "Include a four-digit year before running trend charts.",
      "Problem values:",
      paste(head(unparseable_review_dates$review_date, 5), collapse = ", ")
    ),
    call. = FALSE
  )
}

trend_reviews <- trend_reviews %>%
  mutate(
    month_start = lubridate::floor_date(date_parsed, "month"),
    quarter_start = lubridate::floor_date(date_parsed, "quarter"),
    quarter_label = paste0(lubridate::year(date_parsed), " Q", lubridate::quarter(date_parsed)),
    year_start = lubridate::floor_date(date_parsed, "year"),
    year_label = as.character(lubridate::year(date_parsed))
  ) %>%
  # Ignore broken dates and reviews that cannot be normalized.
  filter(!is.na(date_parsed), !is.na(score_afinn_per_median_review))

# Give every calendar month a simple number: 1, 2, 3, and so on.
# Some months have no reviews. We still include those empty months so a
# "6-month rolling average" really means six calendar months, not six reviewed
# months that may be far apart.
month_lookup <- tibble(
  month_start = seq(
    min(trend_reviews$month_start),
    max(trend_reviews$month_start),
    by = "1 month"
  )
) %>%
  mutate(
    month_index = row_number(),
    month_label = format(month_start, "%b %Y"),
    quarter_start = lubridate::floor_date(month_start, "quarter"),
    quarter_label = paste0(lubridate::year(month_start), " Q", lubridate::quarter(month_start))
  )

trend_reviews <- trend_reviews %>%
  left_join(month_lookup, by = c("month_start", "quarter_start", "quarter_label"))

# These values help us place small statistic labels below some charts.
# score_min and score_max find the lowest and highest sentiment scores.
# score_range is the distance between them.
# We subtract a small part of that range to create empty space below the chart.
score_min <- min(trend_reviews$score_afinn)
score_max <- max(trend_reviews$score_afinn)
score_range <- max(score_max - score_min, 1)
stats_label_y <- score_min - score_range * 0.13
stats_axis_floor <- stats_label_y - score_range * 0.08
year_stats_label_y <- score_min - score_range * 0.18
year_stats_axis_floor <- year_stats_label_y - score_range * 0.24

# This helper adds a "prior average" to a monthly, quarterly, or yearly table.
# "Prior average" means: the average sentiment before the current period starts.
# Example:
# - For 2020, it uses all reviews before 2020.
# - For March 2024, it uses all reviews before March 2024.
# This lets us compare the current period against the hotel's previous history,
# instead of comparing it against an average that includes future reviews.
add_prior_period_average <- function(period_summary) {
  period_summary %>%
    mutate(
      cumulative_score_before = lag(cumsum(score_total)),
      cumulative_count_before = lag(cumsum(review_count)),
      prior_avg = cumulative_score_before / cumulative_count_before,
      stats_label = paste0("avg ", round(period_avg, 1), "\nn=", review_count)
    )
}

# This helper rounds numbers to one decimal place.
# We use it for labels so the chart says "17.5" instead of a long number like
# "17.5748031496063".
format_stat <- function(value) {
  format(round(value, 1), nsmall = 1, trim = TRUE)
}

# First count only the months that actually have reviews.
reviewed_month_averages <- trend_reviews %>%
  group_by(month_start, month_index, month_label, quarter_start, quarter_label) %>%
  summarise(
    period_avg = mean(score_afinn, na.rm = TRUE),
    period_median = median(score_afinn, na.rm = TRUE),
    period_trimmed_mean = calculate_trimmed_mean(score_afinn),
    period_q1 = quantile(score_afinn, 0.25, names = FALSE, na.rm = TRUE),
    period_q3 = quantile(score_afinn, 0.75, names = FALSE, na.rm = TRUE),
    period_min = min(score_afinn, na.rm = TRUE),
    period_max = max(score_afinn, na.rm = TRUE),
    review_count = n(),
    score_total = sum(score_afinn, na.rm = TRUE),
    .groups = "drop"
  )

# Create one row per calendar month.
# period_avg is the average sentiment score scaled to a median-length review.
# review_count tells us how many reviews were written in that month.
# score_total is used for rolling averages and prior averages.
month_averages <- month_lookup %>%
  left_join(
    reviewed_month_averages,
    by = c("month_start", "month_index", "month_label", "quarter_start", "quarter_label")
  ) %>%
  mutate(
    review_count = replace_na(review_count, 0L),
    score_total = replace_na(score_total, 0)
  ) %>%
  arrange(month_start) %>%
  add_prior_period_average() %>%
  mutate(
    # rolling_avg_6 is the 6-month rolling average.
    # For each month, it averages that month plus the previous five months.
    # This smooths out noisy spikes from months with only a few reviews.
    rolling_avg_6 = purrr::map_dbl(
      row_number(),
      ~ {
        window_start <- max(1, .x - 5)
        window_review_count <- sum(review_count[window_start:.x])
        if (window_review_count == 0) {
          NA_real_
        } else {
          sum(score_total[window_start:.x]) / window_review_count
        }
      }
    ),
    # These fields are used for the heatmap.
    # year_label becomes the heatmap row.
    # month_name becomes the heatmap column.
    # heatmap_label is the text printed inside each heatmap cell.
    year_label = as.character(lubridate::year(month_start)),
    month_name = factor(
      month.abb[lubridate::month(month_start)],
      levels = month.abb
    ),
    heatmap_label = if_else(
      review_count > 0,
      paste0(format_stat(period_avg), "\nn=", review_count),
      "n=0"
    )
  )

# quarter_bands stores the start and end position of each quarter on the monthly
# timeline. The rolling chart uses these bands as a light background so the
# viewer can quickly see where each quarter begins and ends.
quarter_bands <- month_lookup %>%
  group_by(quarter_start, quarter_label) %>%
  summarise(
    xmin = min(month_index) - 0.5,
    xmax = max(month_index) + 0.5,
    .groups = "drop"
  ) %>%
  mutate(band = row_number() %% 2)

# Create one row per quarter for the quarterly boxplot.
# We keep the quarter average, review count, and prior average so the chart can
# compare each quarter with the hotel's history before that quarter.
quarter_averages <- trend_reviews %>%
  group_by(quarter_start, quarter_label) %>%
  summarise(
    period_avg = mean(score_afinn, na.rm = TRUE),
    period_median = median(score_afinn, na.rm = TRUE),
    period_trimmed_mean = calculate_trimmed_mean(score_afinn),
    period_q1 = quantile(score_afinn, 0.25, names = FALSE, na.rm = TRUE),
    period_q3 = quantile(score_afinn, 0.75, names = FALSE, na.rm = TRUE),
    period_min = min(score_afinn, na.rm = TRUE),
    period_max = max(score_afinn, na.rm = TRUE),
    review_count = n(),
    score_total = sum(score_afinn, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(quarter_start) %>%
  mutate(quarter_index = row_number()) %>%
  add_prior_period_average() %>%
  left_join(quarter_bands, by = c("quarter_start", "quarter_label"))

# Create one row per year for the yearly boxplot.
# This table contains more statistics because the yearly chart has enough room
# to print a detailed statistics block below each box.
year_averages <- trend_reviews %>%
  group_by(year_label) %>%
  summarise(
    # Store all yearly scores temporarily so we can count outliers after
    # calculating the boxplot fences.
    scores = list(score_afinn),
    period_avg = mean(score_afinn, na.rm = TRUE),
    period_median = median(score_afinn, na.rm = TRUE),
    period_trimmed_mean = calculate_trimmed_mean(score_afinn),
    period_q1 = quantile(score_afinn, 0.25, names = FALSE, na.rm = TRUE),
    period_q3 = quantile(score_afinn, 0.75, names = FALSE, na.rm = TRUE),
    period_min = min(score_afinn, na.rm = TRUE),
    period_max = max(score_afinn, na.rm = TRUE),
    iqr = IQR(score_afinn),
    review_count = n(),
    score_total = sum(score_afinn, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(year_label) %>%
  mutate(year_index = row_number()) %>%
  add_prior_period_average() %>%
  mutate(
    lower_outlier_fence = period_q1 - 1.5 * iqr,
    upper_outlier_fence = period_q3 + 1.5 * iqr
  ) %>%
  # A boxplot normally treats very low or very high values as outliers when
  # they are more than 1.5 IQR away from the box.
  # Here we count those outliers manually so the students can see the number.
  rowwise() %>%
  mutate(n_outliers = sum(scores < lower_outlier_fence | scores > upper_outlier_fence)) %>%
  ungroup() %>%
  select(-stats_label) %>%
  select(-scores) %>%
  mutate(
    stats_label = paste0(
      "n=", review_count,
      "\navg=", format_stat(period_avg),
      "\nmedian=", format_stat(period_median),
      "\ntrimmed=", format_stat(period_trimmed_mean),
      "\nq1=", format_stat(period_q1),
      "\nq3=", format_stat(period_q3),
      "\nmin=", format_stat(period_min),
      "\nmax=", format_stat(period_max),
      "\nn_outliers=", n_outliers
    )
  )

# Build a CSV table that shows how each period compares with the hotel's past.
# This is not a management action rule. It is a statistical monitoring output:
# negative robust z-scores mean the current period is below the historical median.
# Require at least 3 reviews in the current period and at least 20 older reviews
# before raising a statistical flag. This avoids overreacting to tiny samples.
minimum_current_reviews_for_drift <- 3
minimum_baseline_reviews_for_drift <- 20

build_sentiment_period_summary <- function(data, period_type_value, period_start_col, period_label_col) {
  # The same function works for months, quarters, and years. The period_start_col
  # and period_label_col arguments tell R which date columns to group by.
  period_rows <- data %>%
    group_by(
      period_start = .data[[period_start_col]],
      period_label = .data[[period_label_col]]
    ) %>%
    # One summary row is created for each time period. We keep raw sentiment
    # scores for auditability and normalized scores for fair period comparisons.
    summarise(
      period_type = period_type_value,
      review_count = n(),
      median_review_word_count = first(median_review_word_count),
      mean_afinn_raw = mean(score_afinn_raw, na.rm = TRUE),
      median_afinn_raw = median(score_afinn_raw, na.rm = TRUE),
      trimmed_mean_afinn_raw = calculate_trimmed_mean(score_afinn_raw),
      mean_afinn_per_median_review = mean(score_afinn_per_median_review, na.rm = TRUE),
      median_afinn_per_median_review = median(score_afinn_per_median_review, na.rm = TRUE),
      trimmed_mean_afinn_per_median_review = calculate_trimmed_mean(score_afinn_per_median_review),
      mean_syuzhet_raw = mean(score_syuzhet_raw, na.rm = TRUE),
      median_syuzhet_raw = median(score_syuzhet_raw, na.rm = TRUE),
      trimmed_mean_syuzhet_raw = calculate_trimmed_mean(score_syuzhet_raw),
      mean_syuzhet_per_median_review = mean(score_syuzhet_per_median_review, na.rm = TRUE),
      median_syuzhet_per_median_review = median(score_syuzhet_per_median_review, na.rm = TRUE),
      trimmed_mean_syuzhet_per_median_review = calculate_trimmed_mean(score_syuzhet_per_median_review),
      mean_rating = mean(rating_number, na.rm = TRUE),
      median_rating = median(rating_number, na.rm = TRUE),
      trimmed_mean_rating = calculate_trimmed_mean(rating_number),
      low_rating_share = mean(rating_number <= 3, na.rm = TRUE),
      negative_text_share = mean(score_afinn_per_median_review < 0, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    arrange(period_start)

  period_rows %>%
    # rowwise() means the calculations below happen one period at a time.
    # That is important because every period has a different historical baseline.
    rowwise() %>%
    mutate(
      # The baseline contains only reviews before the current period. This prevents
      # future reviews from influencing a past period's warning score.
      baseline_review_count = sum(data$date_parsed < period_start & !is.na(data$score_afinn_per_median_review)),
      historical_median_afinn_per_median_review = if_else(
        baseline_review_count >= minimum_baseline_reviews_for_drift,
        median(data$score_afinn_per_median_review[data$date_parsed < period_start], na.rm = TRUE),
        NA_real_
      ),
      historical_mad_afinn_per_median_review = calculate_robust_mad(
        data$score_afinn_per_median_review[data$date_parsed < period_start],
        minimum_count = minimum_baseline_reviews_for_drift
      ),
      robust_z_afinn_median = calculate_robust_z(
        median_afinn_per_median_review,
        data$score_afinn_per_median_review[data$date_parsed < period_start],
        minimum_count = minimum_baseline_reviews_for_drift
      ),
      # Rating drift uses the same prior-history idea as sentiment drift, but it
      # looks at TripAdvisor star ratings instead of text sentiment.
      historical_median_rating = if_else(
        baseline_review_count >= minimum_baseline_reviews_for_drift,
        median(data$rating_number[data$date_parsed < period_start], na.rm = TRUE),
        NA_real_
      ),
      historical_mad_rating = calculate_robust_mad(
        data$rating_number[data$date_parsed < period_start],
        minimum_count = minimum_baseline_reviews_for_drift
      ),
      robust_z_rating_median = calculate_robust_z(
        median_rating,
        data$rating_number[data$date_parsed < period_start],
        minimum_count = minimum_baseline_reviews_for_drift
      ),
      # A TRUE flag means "this period deserves closer inspection." It does not
      # mean the project has proven a management or investment decision.
      sentiment_drift_flag = review_count >= minimum_current_reviews_for_drift &
        baseline_review_count >= minimum_baseline_reviews_for_drift &
        !is.na(robust_z_afinn_median) &
        robust_z_afinn_median <= -2,
      rating_drift_flag = review_count >= minimum_current_reviews_for_drift &
        baseline_review_count >= minimum_baseline_reviews_for_drift &
        !is.na(robust_z_rating_median) &
        robust_z_rating_median <= -2,
      any_drift_flag = sentiment_drift_flag | rating_drift_flag
    ) %>%
    ungroup()
}

sentiment_period_summary <- bind_rows(
  build_sentiment_period_summary(trend_reviews, "month", "month_start", "month_label"),
  build_sentiment_period_summary(trend_reviews, "quarter", "quarter_start", "quarter_label"),
  build_sentiment_period_summary(trend_reviews, "year", "year_start", "year_label")
)

# Save output/reports/sentiment_period_summary.csv for review and later writing.
write_csv(sentiment_period_summary, sentiment_period_summary_path)

# Add quarter_index and year_index back onto the review-level data.
# The boxplots use these simple numbers for clean x-axis positioning.
trend_reviews <- trend_reviews %>%
  left_join(select(quarter_averages, quarter_start, quarter_index), by = "quarter_start") %>%
  left_join(select(year_averages, year_label, year_index), by = "year_label")

# The monthly timeline has many months, so we show only every 6th month label.
# This keeps the x-axis readable.
month_breaks <- month_lookup$month_index[seq(1, nrow(month_lookup), by = 6)]
month_labels <- month_lookup$month_label[seq(1, nrow(month_lookup), by = 6)]
month_stat_labels <- month_averages %>%
  filter(month_index %in% month_breaks)

# The quarterly timeline has many quarters, so we label every 4th quarter.
# That is roughly one label per year.
quarter_breaks <- quarter_averages$quarter_index[seq(1, nrow(quarter_averages), by = 4)]
quarter_labels <- quarter_averages$quarter_label[seq(1, nrow(quarter_averages), by = 4)]

# Monthly heatmap for a compact month-by-year overview.
# In this chart:
# - Rows are years.
# - Columns are months.
# - Darker green means a higher average sentiment score.
# - Pale or red cells mean lower sentiment.
# - The text inside each cell shows the average score and number of reviews.
p4_heatmap <- ggplot(month_averages, aes(x = month_name, y = year_label, fill = period_avg)) +
  geom_tile(color = "white", linewidth = 0.4) +
  geom_text(aes(label = heatmap_label), size = 2.1, color = "#2C3E50", lineheight = 0.85) +
  scale_fill_gradient2(
    low = "#C0392B",
    mid = "#F7F9F9",
    high = "#16A085",
    midpoint = 0,
    name = "Average\nAFINN per\nmedian-length review",
    na.value = "#F4F6F7"
  ) +
  labs(
    title = "Monthly Sentiment Heatmap (AFINN per Median-Length Review)",
    subtitle = "Each tile shows monthly average length-normalized sentiment and review count",
    x = "Month",
    y = "Year"
  ) +
  theme_premium() +
  theme(
    legend.position = "right",
    axis.text.x = element_text(angle = 45, hjust = 1)
  )

show_plot_for_interactive_use(p4_heatmap)

ggsave(file.path(figures_dir, "sentiment_trend_monthly_heatmap.png"), plot = p4_heatmap, width = 10, height = 8, dpi = 300)

# Monthly rolling chart for the trend story.
# This chart is better than 189 tiny monthly boxplots because many months have
# very few reviews. Instead, it shows:
# - A grey line for the raw monthly average.
# - Circles for each monthly average.
# - Larger circles when a month has more reviews.
# - A blue 6-month rolling average to smooth the noise.
# - A dotted prior average line to compare the current month with past history.
p4_rolling <- ggplot() +
  geom_rect(
    data = quarter_bands,
    aes(xmin = xmin, xmax = xmax, ymin = -Inf, ymax = Inf, fill = factor(band)),
    alpha = 0.08,
    inherit.aes = FALSE
  ) +
  geom_line(
    data = month_averages,
    aes(x = month_index, y = period_avg),
    color = "#95A5A6",
    linewidth = 0.45,
    alpha = 0.7,
    inherit.aes = FALSE
  ) +
  geom_point(
    data = month_averages,
    aes(x = month_index, y = period_avg, size = review_count, color = period_avg),
    shape = 21,
    fill = "white",
    stroke = 0.9,
    alpha = 0.9,
    inherit.aes = FALSE
  ) +
  geom_line(
    data = month_averages,
    aes(x = month_index, y = rolling_avg_6),
    color = "#2980B9",
    linewidth = 1.1,
    inherit.aes = FALSE
  ) +
  geom_line(
    data = filter(month_averages, !is.na(prior_avg)),
    aes(x = month_index, y = prior_avg),
    color = "#7F8C8D",
    linetype = "dotted",
    linewidth = 1,
    inherit.aes = FALSE
  ) +
  scale_fill_manual(values = c("0" = "#EAF2F8", "1" = "#FDF2E9")) +
  scale_color_gradient2(low = "#C0392B", mid = "#95A5A6", high = "#16A085", midpoint = 0) +
  scale_size_continuous(range = c(1.4, 5)) +
  scale_x_continuous(breaks = month_breaks, labels = month_labels) +
  labs(
    title = "Monthly Sentiment Trend with Rolling Average (AFINN per Median-Length Review)",
    subtitle = "Points show monthly averages sized by review count; blue line is the 6-month rolling average; dotted line is prior average",
    x = "Timeline (Month/Year)",
    y = "AFINN per median-length review"
  ) +
  theme_premium() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

# Show the chart in RStudio, but do not open a PDF device during Rscript runs.
show_plot_for_interactive_use(p4_rolling)

ggsave(file.path(figures_dir, "sentiment_trend_monthly_rolling.png"), plot = p4_rolling, width = 10, height = 5.5, dpi = 300)

# Monthly robust drift monitor. This chart saves output/figures/sentiment_drift_monitor.png.
# It views each month's median sentiment
# against only the reviews that came before that month. A value near 0 means the
# month looks typical. A value below -2 means it is unusually low by a robust
# median-and-MAD rule, if there are enough current and historical reviews.
monthly_drift_monitor <- sentiment_period_summary %>%
  filter(period_type == "month", !is.na(robust_z_afinn_median))

if (nrow(monthly_drift_monitor) > 0) {
  p4_drift <- ggplot(
    monthly_drift_monitor,
    aes(x = period_start, y = robust_z_afinn_median)
  ) +
    geom_hline(yintercept = 0, color = "#7F8C8D", linewidth = 0.45) +
    geom_hline(yintercept = -2, color = "#C0392B", linetype = "dashed", linewidth = 0.8) +
    geom_col(aes(fill = any_drift_flag), width = 24, alpha = 0.76) +
    geom_point(aes(size = review_count), color = "#2C3E50", alpha = 0.82) +
    scale_fill_manual(values = c("FALSE" = "#7FB3D5", "TRUE" = "#C0392B")) +
    scale_size_continuous(range = c(1.8, 5.2)) +
    scale_x_date(date_breaks = "1 year", date_labels = "%Y") +
    labs(
      title = "Monthly Sentiment Drift Monitor",
      subtitle = "Monthly median AFINN compared with prior reviews using historical median and MAD",
      x = "Review month",
      y = "Robust z-score"
    ) +
    theme_premium() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1))

  show_plot_for_interactive_use(p4_drift)
  ggsave(sentiment_drift_monitor_path, plot = p4_drift, width = 10, height = 5.5, dpi = 300)
} else {
  save_placeholder_plot(
    basename(sentiment_drift_monitor_path),
    "Monthly Sentiment Drift Monitor",
    "There are not enough historical reviews to calculate robust drift scores yet.",
    width = 10,
    height = 5.5
  )
}

# Quarterly boxplot for quarter-level statistics.
# A quarterly boxplot groups all reviews from the same quarter together.
# This is useful because quarters usually have more reviews than single months,
# so the distribution is more meaningful.
p5 <- ggplot(trend_reviews, aes(x = quarter_index, y = score_afinn, group = quarter_index)) +
  geom_boxplot(fill = "#D5F5E3", color = "#1E8449", alpha = 0.85, outlier.alpha = 0.35) +
  geom_point(
    data = quarter_averages,
    aes(x = quarter_index, y = period_avg),
    shape = 23,
    size = 2.8,
    fill = "#C0392B",
    color = "white",
    inherit.aes = FALSE
  ) +
  geom_line(
    data = filter(quarter_averages, !is.na(prior_avg)),
    aes(x = quarter_index, y = prior_avg),
    color = "#7F8C8D",
    linetype = "dotted",
    linewidth = 1,
    inherit.aes = FALSE
  ) +
  geom_text(
    data = quarter_averages,
    aes(x = quarter_index, y = stats_label_y, label = stats_label),
    angle = 90,
    hjust = 0.5,
    size = 1.9,
    color = "#34495E",
    inherit.aes = FALSE
  ) +
  scale_x_continuous(breaks = quarter_breaks, labels = quarter_labels) +
  scale_y_continuous(limits = c(stats_axis_floor, NA), expand = expansion(mult = c(0, 0.12))) +
  labs(
    title = "Quarterly Sentiment Distribution (AFINN per Median-Length Review)",
    subtitle = "Each box summarizes one quarter; labels show quarter avg and n; dotted line shows the average before each quarter",
    x = "Quarter",
    y = "AFINN per median-length review"
  ) +
  theme_premium() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7))

show_plot_for_interactive_use(p5)
ggsave(file.path(figures_dir, "sentiment_trend_quarterly.png"), plot = p5, width = 14, height = 6, dpi = 300)

# Yearly boxplot for year-level statistics.
# The yearly chart has the most detailed labels because there are only about
# twenty years, which leaves enough room below each box.
# The label under each year shows:
# n = number of reviews
# avg = average sentiment score
# median = middle sentiment score
# q1 and q3 = the lower and upper edges of the box
# min and max = lowest and highest sentiment scores
# n_outliers = how many unusual values fall outside the whiskers
p6 <- ggplot(trend_reviews, aes(x = year_index, y = score_afinn, group = year_index)) +
  geom_boxplot(fill = "#FADBD8", color = "#922B21", alpha = 0.85, outlier.alpha = 0.35) +
  geom_point(
    data = year_averages,
    aes(x = year_index, y = period_avg),
    shape = 23,
    size = 3,
    fill = "#2C3E50",
    color = "white",
    inherit.aes = FALSE
  ) +
  geom_text(
    data = year_averages,
    aes(x = year_index, y = year_stats_label_y, label = stats_label),
    vjust = 0.5,
    size = 2,
    lineheight = 0.86,
    color = "#2C3E50",
    inherit.aes = FALSE
  ) +
  geom_line(
    data = filter(year_averages, !is.na(prior_avg)),
    aes(x = year_index, y = prior_avg),
    color = "#7F8C8D",
    linetype = "dotted",
    linewidth = 1,
    inherit.aes = FALSE
  ) +
  scale_x_continuous(breaks = year_averages$year_index, labels = year_averages$year_label) +
  scale_y_continuous(limits = c(year_stats_axis_floor, NA), expand = expansion(mult = c(0, 0.12))) +
  labs(
    title = "Yearly Sentiment Distribution (AFINN per Median-Length Review)",
    subtitle = "Each box summarizes one year; labels show n, mean, median, quartiles, min, max, and outliers",
    x = "Year",
    y = "AFINN per median-length review"
  ) +
  theme_premium() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

show_plot_for_interactive_use(p6)
ggsave(file.path(figures_dir, "sentiment_trend_yearly.png"), plot = p6, width = 16, height = 8, dpi = 300)

if (interactive()) {
  cat("Showing the emotions breakdown chart again in the RStudio Plots pane.\n")
  print(p2)
}

cat("Visualizations successfully drawn and saved to the 'output/figures/' folder!\n")

## Step 5: Aspect Text Analysis

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 5: Aspect Text Analysis
# =====================================================================
# PURPOSE OF THIS SCRIPT:
# This script connects the structured TripAdvisor aspect ratings to the review
# text. The aspect ratings are used as weak labels: if a review gives "Value"
# a low score, the review text is treated as evidence associated with a value
# problem. This does not prove that every word in the review is about value,
# but it helps us find the words, phrases, and examples that deserve reading.

In [ ]:
# =====================================================================
# STEP 1: Load Packages and Data
# =====================================================================
# These packages are collections of ready-made tools:
# - tidyverse helps us clean tables, calculate summaries, and draw charts.
# - tidytext helps us split review text into words and short phrases.
# - syuzhet gives us the sentiment dictionaries used earlier in the project.
library(tidyverse)
library(tidytext)
library(syuzhet)

# This file stores the folder and file names used throughout the project.
# Using shared paths keeps every script pointed at the same inputs and outputs.
source("scripts/data_config.R")
source("scripts/helpers.R")

# Step 5 depends on the scored review table from Step 3 and the token table
# from Step 2. Before doing any analysis, we check that both files exist so
# beginners get a clear message instead of a confusing R error later.
required_paths <- c(sentiment_scores_path, cleaned_tokens_path)
missing_paths <- required_paths[!file.exists(required_paths)]
if (length(missing_paths) > 0) {
  stop(
    paste(
      "Missing required analysis data:",
      paste(missing_paths, collapse = ", "),
      "Run Steps 2 and 3 before aspect text analysis."
    ),
    call. = FALSE
  )
}

# research_data is one row per review, including ratings and sentiment scores.
# cleaned_tokens is one row per important word from the reviews.
research_data <- read_csv(sentiment_scores_path, show_col_types = FALSE)
cleaned_tokens <- read_csv(cleaned_tokens_path, show_col_types = FALSE)

# Create the output folders if they are missing. If they already exist, R keeps
# going quietly because showWarnings = FALSE.
dir.create(figures_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(reports_dir, recursive = TRUE, showWarnings = FALSE)

cat("Aspect text analysis data loaded successfully!\n")

In [ ]:
# =====================================================================
# STEP 2: Shared Settings and Helper Functions
# =====================================================================
# A function is a reusable recipe. This one stores the chart styling choices so
# all Step 5 charts have the same fonts, colors, and spacing.
theme_premium <- function() {
  theme_minimal() +
    theme(
      text = element_text(color = "#2C3E50"),
      plot.title = element_text(face = "bold", size = 16, hjust = 0.5, color = "#1A252C"),
      plot.subtitle = element_text(size = 11, hjust = 0.5, color = "#5A6F7C"),
      axis.title = element_text(face = "bold", size = 11),
      legend.position = "bottom",
      panel.grid.minor = element_blank(),
      strip.text = element_text(face = "bold", color = "#2C3E50")
    )
}

# RStudio has an interactive Plots pane, but Rscript runs without one.
# This helper only prints charts when a person is using R interactively.
# The charts are still saved as PNG files by ggsave() in every run.
show_plot_for_interactive_use <- function(plot_object) {
  if (interactive()) {
    print(plot_object)
  }
}

# Some sampled datasets are small enough that a table may be missing an
# expected numeric column, such as count_low when no low-score reviews exist.
# This helper adds the missing column filled with zeroes before later math runs.
add_missing_numeric_column <- function(data, column_name) {
  if (!column_name %in% names(data)) {
    data[[column_name]] <- 0
  }

  data
}

# When a sample has too little data for a chart, we still save a PNG file. That
# keeps the workflow predictable and tells beginners why the chart is empty.
save_placeholder_plot <- function(file_name, title, message, width = 10, height = 6) {
  placeholder_plot <- ggplot() +
    annotate(
      "text",
      x = 0,
      y = 0,
      label = message,
      size = 4,
      color = "#2C3E50",
      lineheight = 0.95
    ) +
    labs(title = title, x = NULL, y = NULL) +
    theme_void() +
    theme(
      plot.title = element_text(face = "bold", size = 15, hjust = 0.5, color = "#1A252C")
    )

  ggsave(file.path(figures_dir, file_name), plot = placeholder_plot, width = width, height = height, dpi = 300)
  show_plot_for_interactive_use(placeholder_plot)
}

# Correlation measures whether two numbers tend to move together.
# This helper returns NA when there are too few complete rows or when all values
# are the same, because correlation is not meaningful in those cases.
safe_correlation <- function(x_values, y_values) {
  complete_rows <- complete.cases(x_values, y_values)
  if (sum(complete_rows) < 3) {
    return(NA_real_)
  }

  x_complete <- x_values[complete_rows]
  y_complete <- y_values[complete_rows]
  if (sd(x_complete) == 0 || sd(y_complete) == 0) {
    return(NA_real_)
  }

  cor(x_complete, y_complete)
}

# Long review text can make CSV reports hard to skim. This helper creates a
# short preview while keeping the full review text in a separate column.
make_review_excerpt <- function(text_values, width = 280) {
  text_values %>%
    as.character() %>%
    str_squish() %>%
    str_trunc(width = width)
}

# These are the structured TripAdvisor aspect-rating columns. The left side is
# the column name in the CSV, and the right side is the friendly chart label.
aspect_rating_columns <- c(
  "value_rating" = "Value",
  "rooms_rating" = "Rooms",
  "location_rating" = "Location",
  "cleanliness_rating" = "Cleanliness",
  "service_rating" = "Service",
  "sleep_quality_rating" = "Sleep quality"
)

# Empty result tables still need the same columns as normal result tables.
# That way write_csv(), chart code, and tests can run even on tiny samples.
empty_term_comparison <- function(term_type) {
  tibble(
    aspect_key = character(),
    aspect_label = factor(character(), levels = unname(aspect_rating_columns)),
    term = character(),
    count_high = numeric(),
    count_low = numeric(),
    total_high = numeric(),
    total_low = numeric(),
    low_review_share = numeric(),
    high_review_share = numeric(),
    lift_low_vs_high = numeric(),
    log_lift_low_vs_high = numeric(),
    starts_with_negation = logical(),
    afinn_term_score = numeric(),
    term_sentiment = character(),
    term_type = rep(as.character(term_type), 0)
  )
}

# Some datasets may not include every optional aspect rating. This line keeps
# only the aspect columns that are actually present in the current data.
available_aspect_columns <- names(aspect_rating_columns)[names(aspect_rating_columns) %in% names(research_data)]

if (length(available_aspect_columns) == 0) {
  stop("No structured aspect rating columns were found in the scored dataset.", call. = FALSE)
}

# These words are too general for our diagnostic tables. For example, "hotel"
# and "stay" are common in almost every review, so they do not help explain why
# a specific aspect received a low score.
domain_neutral_words <- c(
  "bali", "bvlgari", "bulgari", "hotel", "resort", "review", "tripadvisor",
  "stay", "stayed", "staying", "room", "rooms", "villa", "villas",
  "service", "property", "place", "guest", "guests", "day", "days",
  "night", "nights", "time", "times", "experience", "experiences"
)

# Give each aspect a stable color so charts are easier to compare.
aspect_colors <- c(
  "Value" = "#E76F51",
  "Rooms" = "#457B9D",
  "Location" = "#2A9D8F",
  "Cleanliness" = "#6D597A",
  "Service" = "#264653",
  "Sleep quality" = "#F4A261"
)

In [ ]:
# =====================================================================
# STEP 3: Prepare Review-by-Aspect Data
# =====================================================================
# The source table has one row per review and six possible aspect columns.
# For this analysis, it is easier to reshape the data into one row per
# review-aspect pair. A single review can therefore become up to six rows:
# one for Value, one for Rooms, one for Location, and so on.
# Start with one row per review. The helper adds raw and normalized sentiment
# columns before we reshape the table into one row per review-aspect pair.
aspect_reviews <- research_data %>%
  prepare_length_normalized_sentiment() %>%
  mutate(
    # Force IDs and numeric fields into predictable types. This prevents joins
    # from failing if one file reads review_id as text and another reads it as
    # a number.
    review_id = as.character(review_id),
    overall_rating = readr::parse_number(as.character(rating)),
    # AFINN and Syuzhet are summed scores, so long reviews can look more
    # emotional simply because they contain more words. We keep raw scores for
    # traceability, then use scores scaled to the median review length for aspect
    # comparisons. This keeps a short review and a long review on the same scale.
    score_afinn_number = score_afinn_per_median_review,
    score_syuzhet_number = score_syuzhet_per_median_review,
    across(all_of(available_aspect_columns), ~ readr::parse_number(as.character(.x)))
  ) %>%
  select(
    review_id,
    review_date,
    title,
    review_text,
    cleaned_text,
    rating,
    overall_rating,
    score_afinn_raw,
    score_afinn_per_median_review,
    score_afinn_number,
    score_syuzhet_raw,
    score_syuzhet_per_median_review,
    score_syuzhet_number,
    sentiment_category,
    review_word_count,
    median_review_word_count,
    anger,
    anticipation,
    disgust,
    fear,
    joy,
    sadness,
    surprise,
    trust,
    negative,
    positive,
    all_of(available_aspect_columns)
  ) %>%
  # pivot_longer stacks the six aspect columns into two columns:
  # aspect_key says which aspect it is, and aspect_rating stores the score.
  pivot_longer(
    cols = all_of(available_aspect_columns),
    names_to = "aspect_key",
    values_to = "aspect_rating"
  ) %>%
  mutate(
    # aspect_label turns column names such as value_rating into labels such as
    # Value. The factor levels keep the aspects in the same order everywhere.
    aspect_label = factor(
      aspect_rating_columns[aspect_key],
      levels = unname(aspect_rating_columns)
    ),
    # We split aspect scores into simple bands:
    # - 1 to 3 means the guest signaled a weaker aspect.
    # - 4 to 5 means the guest gave that aspect a stronger score.
    band_key = case_when(
      !is.na(aspect_rating) & aspect_rating <= 3 ~ "low",
      !is.na(aspect_rating) & aspect_rating >= 4 ~ "high",
      TRUE ~ NA_character_
    ),
    aspect_score_band = case_when(
      band_key == "low" ~ "Low aspect score (1-3)",
      band_key == "high" ~ "High aspect score (4-5)",
      TRUE ~ NA_character_
    ),
    aspect_score_band = factor(
      aspect_score_band,
      levels = c("Low aspect score (1-3)", "High aspect score (4-5)")
    )
  )

# This smaller index keeps only the columns needed to connect each review to
# its low/high aspect band. We use distinct() so the same review-aspect pair is
# counted only once.
aspect_review_index <- aspect_reviews %>%
  filter(!is.na(band_key)) %>%
  distinct(aspect_key, aspect_label, review_id, band_key, aspect_score_band)

# Count how many reviews are in the low and high groups for each aspect.
# These totals are needed later when we calculate whether a word is unusually
# common in low-score reviews.
aspect_band_totals <- aspect_review_index %>%
  count(aspect_key, aspect_label, band_key, name = "reviews_in_band") %>%
  pivot_wider(
    names_from = band_key,
    values_from = reviews_in_band,
    values_fill = 0,
    names_prefix = "total_"
  ) %>%
  add_missing_numeric_column("total_low") %>%
  add_missing_numeric_column("total_high")

In [ ]:
# =====================================================================
# STEP 4: Aspect-Specific Sentiment Alignment
# =====================================================================
# This summary asks: when an aspect rating is lower, does the full review text
# also look less positive? It combines structured aspect ratings, overall star
# ratings, length-normalized AFINN/Syuzhet sentiment, and NRC emotion counts.
aspect_text_alignment <- aspect_reviews %>%
  filter(!is.na(aspect_rating)) %>%
  group_by(aspect_key, aspect_label) %>%
  summarise(
    reviews_with_aspect_rating = n_distinct(review_id),
    mean_aspect_rating = mean(aspect_rating, na.rm = TRUE),
    median_aspect_rating = median(aspect_rating, na.rm = TRUE),
    low_score_count = sum(aspect_rating <= 3, na.rm = TRUE),
    low_score_share = low_score_count / reviews_with_aspect_rating,
    mean_overall_rating = mean(overall_rating, na.rm = TRUE),
    mean_afinn_sentiment = mean(score_afinn_number, na.rm = TRUE),
    median_afinn_sentiment = median(score_afinn_number, na.rm = TRUE),
    mean_syuzhet_sentiment = mean(score_syuzhet_number, na.rm = TRUE),
    negative_text_count = sum(score_afinn_number < 0, na.rm = TRUE),
    negative_text_share = mean(score_afinn_number < 0, na.rm = TRUE),
    mean_anger = mean(anger, na.rm = TRUE),
    mean_negative_nrc = mean(negative, na.rm = TRUE),
    corr_aspect_to_afinn = safe_correlation(aspect_rating, score_afinn_number),
    corr_aspect_to_overall_rating = safe_correlation(aspect_rating, overall_rating),
    .groups = "drop"
  ) %>%
  arrange(mean_aspect_rating)

write_csv(
  aspect_text_alignment %>%
    mutate(
      across(
        c(
          mean_aspect_rating,
          median_aspect_rating,
          low_score_share,
          mean_overall_rating,
          mean_afinn_sentiment,
          median_afinn_sentiment,
          mean_syuzhet_sentiment,
          negative_text_share,
          mean_anger,
          mean_negative_nrc,
          corr_aspect_to_afinn,
          corr_aspect_to_overall_rating
        ),
        ~ round(.x, 3)
      )
    ),
  file.path(reports_dir, "aspect_text_alignment.csv")
)

# The alignment table above gives one row per aspect. This table gives two rows
# per aspect: one for low aspect scores and one for high aspect scores. That
# makes it easier to compare low-score reviews against high-score reviews.
aspect_text_band_summary <- aspect_reviews %>%
  filter(!is.na(aspect_score_band)) %>%
  group_by(aspect_key, aspect_label, band_key, aspect_score_band) %>%
  summarise(
    review_count = n_distinct(review_id),
    mean_aspect_rating = mean(aspect_rating, na.rm = TRUE),
    mean_overall_rating = mean(overall_rating, na.rm = TRUE),
    mean_afinn_sentiment = mean(score_afinn_number, na.rm = TRUE),
    median_afinn_sentiment = median(score_afinn_number, na.rm = TRUE),
    negative_text_share = mean(score_afinn_number < 0, na.rm = TRUE),
    mean_anger = mean(anger, na.rm = TRUE),
    mean_negative_nrc = mean(negative, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(aspect_label, band_key)

write_csv(
  aspect_text_band_summary %>%
    mutate(
      across(
        c(
          mean_aspect_rating,
          mean_overall_rating,
          mean_afinn_sentiment,
          median_afinn_sentiment,
          negative_text_share,
          mean_anger,
          mean_negative_nrc
        ),
        ~ round(.x, 3)
      )
    ),
  file.path(reports_dir, "aspect_text_band_summary.csv")
)

In [ ]:
# =====================================================================
# STEP 5: Aspect-Conditioned Text Mining
# =====================================================================
# Here we prepare single words for comparison. Each review-word combination is
# kept once, so a word repeated 20 times in the same review does not dominate
# the result by itself.
token_review_data <- cleaned_tokens %>%
  transmute(
    review_id = as.character(review_id),
    term = str_to_lower(as.character(word))
  ) %>%
  filter(
    !is.na(term),
    str_detect(term, "^[a-z]+$"),
    nchar(term) >= 3,
    !term %in% domain_neutral_words
  ) %>%
  distinct(review_id, term)

data("stop_words", package = "tidytext")
negation_words <- c("no", "not", "never", "without", "cannot")
phrase_stop_words <- setdiff(stop_words$word, negation_words)

# Phrases can explain problems better than isolated words. This creates
# two-word phrases such as "airport transfer" or "poorly maintained", then
# removes phrases made from stopwords or generic hotel words. Negation words
# are allowed because phrases such as "not worth" and "no apology" are useful
# complaint evidence.
phrase_review_data <- research_data %>%
  transmute(
    review_id = as.character(review_id),
    cleaned_text = as.character(cleaned_text)
  ) %>%
  filter(!is.na(cleaned_text), cleaned_text != "") %>%
  unnest_tokens(term, cleaned_text, token = "ngrams", n = 2) %>%
  separate(term, into = c("word_one", "word_two"), sep = " ", remove = FALSE, fill = "right") %>%
  filter(
    !is.na(word_one),
    !is.na(word_two),
    str_detect(word_one, "^[a-z]+$"),
    str_detect(word_two, "^[a-z]+$"),
    # Keep short negation words such as "no" because phrases like "no apology"
    # are meaningful complaints even though the first word has only two letters.
    nchar(word_one) >= 3 | word_one %in% negation_words,
    nchar(word_two) >= 3,
    !word_one %in% phrase_stop_words,
    !word_two %in% phrase_stop_words,
    !word_one %in% domain_neutral_words,
    !word_two %in% domain_neutral_words
  ) %>%
  distinct(review_id, term)

# This reusable function compares text in low-score reviews against text in
# high-score reviews for each aspect.
#
# The key idea is "lift":
# - If a word appears often in low Value reviews but rarely in high Value
#   reviews, that word has high low-score lift for Value.
# - If a word appears equally often in low and high reviews, it is less useful
#   for diagnosis.
build_term_comparison <- function(term_review_data, term_type) {
  joined_terms <- aspect_review_index %>%
    # Many-to-many is expected here because one review can have many aspect
    # rows and many words or phrases.
    inner_join(term_review_data, by = "review_id", relationship = "many-to-many")

  # A very small sample may have no matching words, no phrases, or only one
  # rating band. Returning an empty table is more useful than stopping the
  # entire project with a missing-column error.
  if (nrow(joined_terms) == 0) {
    return(empty_term_comparison(term_type))
  }

  joined_terms %>%
    distinct(aspect_key, aspect_label, band_key, review_id, term) %>%
    count(aspect_key, aspect_label, band_key, term, name = "review_count") %>%
    pivot_wider(
      names_from = band_key,
      values_from = review_count,
      values_fill = 0,
      names_prefix = "count_"
    ) %>%
    add_missing_numeric_column("count_low") %>%
    add_missing_numeric_column("count_high") %>%
    left_join(aspect_band_totals, by = c("aspect_key", "aspect_label")) %>%
    add_missing_numeric_column("total_low") %>%
    add_missing_numeric_column("total_high") %>%
    mutate(
      count_low = coalesce(count_low, 0L),
      count_high = coalesce(count_high, 0L),
      total_low = coalesce(total_low, 0L),
      total_high = coalesce(total_high, 0L),
      # Add 0.5 to the counts and 1 to the totals. This is a standard smoothing
      # trick that avoids division by zero when a word appears in one group but
      # never appears in the comparison group.
      low_review_share = (count_low + 0.5) / (total_low + 1),
      high_review_share = (count_high + 0.5) / (total_high + 1),
      lift_low_vs_high = low_review_share / high_review_share,
      # The log version makes very large ratios easier to chart and compare.
      log_lift_low_vs_high = log(lift_low_vs_high),
      # Use AFINN to color words as positive, negative, or neutral. For phrases
      # that start with a negation word, mark the phrase as negative so "not
      # worth" is not colored as positive just because "worth" is positive.
      starts_with_negation = term_type == "phrase" &
        str_detect(term, paste0("^(", paste(negation_words, collapse = "|"), ")\\s+")),
      afinn_term_score = syuzhet::get_sentiment(term, method = "afinn"),
      afinn_term_score = if_else(
        starts_with_negation & afinn_term_score > 0,
        -afinn_term_score,
        afinn_term_score
      ),
      term_sentiment = case_when(
        starts_with_negation ~ "negative",
        afinn_term_score > 0 ~ "positive",
        afinn_term_score < 0 ~ "negative",
        TRUE ~ "neutral"
      ),
      term_type = term_type
    ) %>%
    filter(count_low >= 2) %>%
    arrange(aspect_label, desc(log_lift_low_vs_high), desc(count_low), term)
}

# Run the same comparison once for single words and once for two-word phrases.
aspect_text_key_terms <- build_term_comparison(token_review_data, "word")
aspect_text_key_phrases <- build_term_comparison(phrase_review_data, "phrase")

write_csv(
  aspect_text_key_terms %>%
    mutate(
      low_review_share = round(low_review_share, 3),
      high_review_share = round(high_review_share, 3),
      lift_low_vs_high = round(lift_low_vs_high, 3),
      log_lift_low_vs_high = round(log_lift_low_vs_high, 3)
    ),
  file.path(reports_dir, "aspect_text_key_terms.csv")
)

write_csv(
  aspect_text_key_phrases %>%
    mutate(
      low_review_share = round(low_review_share, 3),
      high_review_share = round(high_review_share, 3),
      lift_low_vs_high = round(lift_low_vs_high, 3),
      log_lift_low_vs_high = round(log_lift_low_vs_high, 3)
    ),
  file.path(reports_dir, "aspect_text_key_phrases.csv")
)

In [ ]:
# =====================================================================
# STEP 6: Aspect-Text Mismatch Review Tables
# =====================================================================
# A mismatch is a review where the signals do not fully agree. These are useful
# for qualitative reading because they often contain nuance that averages hide.
aspect_text_mismatches <- aspect_reviews %>%
  filter(!is.na(aspect_rating)) %>%
  mutate(
    # Example: a guest gives 5 stars overall but gives Value a 2.
    high_overall_low_aspect = !is.na(overall_rating) & overall_rating >= 4 & aspect_rating <= 3,
    # Example: a guest gives a low aspect score, but the whole review still has
    # positive sentiment. This can happen when the problem is specific.
    low_aspect_positive_text = aspect_rating <= 3 & !is.na(score_afinn_number) & score_afinn_number > 0,
    # Example: the structured aspect score is high, but the text has negative
    # sentiment. This may point to a topic not captured by the aspect field.
    high_aspect_negative_text = aspect_rating >= 4 & !is.na(score_afinn_number) & score_afinn_number < 0,
    # Example: the overall stay rating is low, but one aspect still performed
    # well. This helps avoid treating every part of the experience as broken.
    low_overall_high_aspect = !is.na(overall_rating) & overall_rating <= 3 & aspect_rating >= 4
  ) %>%
  filter(high_overall_low_aspect | low_aspect_positive_text | high_aspect_negative_text | low_overall_high_aspect) %>%
  mutate(
    # pmap_chr() builds the label one review at a time, and it also works when
    # the table has zero rows. That keeps tiny samples from crashing here.
    mismatch_types = pmap_chr(
      list(
        high_overall_low_aspect,
        low_aspect_positive_text,
        high_aspect_negative_text,
        low_overall_high_aspect
      ),
      function(high_overall_low_aspect, low_aspect_positive_text, high_aspect_negative_text, low_overall_high_aspect) {
        paste(
          c(
            if (isTRUE(high_overall_low_aspect)) "high overall rating with low aspect rating",
            if (isTRUE(low_aspect_positive_text)) "low aspect rating with positive text sentiment",
            if (isTRUE(high_aspect_negative_text)) "high aspect rating with negative text sentiment",
            if (isTRUE(low_overall_high_aspect)) "low overall rating with high aspect rating"
          ),
          collapse = "; "
        )
      }
    )
  ) %>%
  ungroup() %>%
  transmute(
    review_id,
    review_date,
    aspect = as.character(aspect_label),
    aspect_rating,
    overall_rating,
    score_afinn_per_median_review = score_afinn_number,
    score_afinn_raw,
    score_syuzhet_per_median_review = score_syuzhet_number,
    score_syuzhet_raw,
    sentiment_category,
    mismatch_types,
    title,
    review_excerpt = make_review_excerpt(review_text),
    review_text
  ) %>%
  arrange(aspect, aspect_rating, score_afinn_per_median_review, review_id)

write_csv(
  aspect_text_mismatches,
  file.path(reports_dir, "aspect_text_mismatches.csv")
)

# Save a smaller table of the lowest aspect-score examples. These are the rows
# a researcher or manager should read first when interpreting the numbers.
aspect_qualitative_examples <- aspect_reviews %>%
  filter(!is.na(aspect_rating), aspect_rating <= 3) %>%
  mutate(
    review_excerpt = make_review_excerpt(review_text)
  ) %>%
  group_by(aspect_label) %>%
  arrange(aspect_rating, score_afinn_number, .by_group = TRUE) %>%
  slice_head(n = 8) %>%
  ungroup() %>%
  transmute(
    aspect = as.character(aspect_label),
    review_id,
    review_date,
    aspect_rating,
    overall_rating,
    score_afinn_per_median_review = score_afinn_number,
    score_afinn_raw,
    score_syuzhet_per_median_review = score_syuzhet_number,
    score_syuzhet_raw,
    sentiment_category,
    title,
    review_excerpt,
    review_text
  )

write_csv(
  aspect_qualitative_examples,
  file.path(reports_dir, "aspect_qualitative_examples.csv")
)

In [ ]:
# =====================================================================
# STEP 7: Visualize Aspect Text Results
# =====================================================================
# This chart checks whether length-normalized text sentiment generally improves
# as the structured aspect rating increases. Each panel is one aspect.
aspect_sentiment_plot_data <- aspect_reviews %>%
  filter(!is.na(aspect_rating), !is.na(score_afinn_number)) %>%
  mutate(aspect_rating_group = factor(aspect_rating, levels = 1:5))

if (nrow(aspect_sentiment_plot_data) > 0) {
  p_aspect_sentiment <- ggplot(
    aspect_sentiment_plot_data,
    aes(x = aspect_rating_group, y = score_afinn_number, fill = aspect_rating_group)
  ) +
    geom_hline(yintercept = 0, color = "#7F8C8D", linewidth = 0.4, linetype = "dashed") +
    geom_boxplot(alpha = 0.82, outlier.alpha = 0.25, width = 0.62) +
    facet_wrap(~ aspect_label, ncol = 3) +
    scale_fill_brewer(palette = "RdYlGn") +
    labs(
      title = "Text Sentiment by Structured Aspect Rating",
      subtitle = "Boxes compare whole-review AFINN scaled to a median-length review for low and high aspect scores",
      x = "Aspect rating",
      y = "AFINN per median-length review"
    ) +
    theme_premium() +
    theme(legend.position = "none")

  show_plot_for_interactive_use(p_aspect_sentiment)
  ggsave(
    file.path(figures_dir, "aspect_sentiment_by_rating_boxplot.png"),
    plot = p_aspect_sentiment,
    width = 11,
    height = 7,
    dpi = 300
  )
} else {
  save_placeholder_plot(
    "aspect_sentiment_by_rating_boxplot.png",
    "Text Sentiment by Structured Aspect Rating",
    "There are no complete aspect rating and text sentiment pairs in this sample.",
    width = 11,
    height = 7
  )
}

# Pick the strongest low-score-associated words for each aspect so the chart is
# readable. The full ranked table is still saved in the CSV report.
plot_key_terms <- aspect_text_key_terms %>%
  group_by(aspect_label) %>%
  slice_max(order_by = log_lift_low_vs_high, n = 6, with_ties = FALSE) %>%
  ungroup() %>%
  mutate(
    term_for_plot = tidytext::reorder_within(term, log_lift_low_vs_high, aspect_label),
    label_text = paste0("low n=", count_low)
  )

# This chart shows the words that most separate low-score reviews from
# high-score reviews for each aspect.
if (nrow(plot_key_terms) > 0) {
  p_low_terms <- ggplot(
    plot_key_terms,
    aes(x = term_for_plot, y = log_lift_low_vs_high, fill = term_sentiment)
  ) +
    geom_col(width = 0.72, alpha = 0.88) +
    geom_text(aes(label = label_text), hjust = -0.08, size = 2.5, color = "#2C3E50") +
    coord_flip() +
    facet_wrap(~ aspect_label, scales = "free_y", ncol = 2) +
    tidytext::scale_x_reordered() +
    scale_fill_manual(
      values = c("negative" = "#E76F51", "neutral" = "#457B9D", "positive" = "#2A9D8F"),
      name = "AFINN word tone"
    ) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.28))) +
    labs(
      title = "Words Most Associated with Low Aspect Scores",
      subtitle = "Bars show smoothed log lift in low-score reviews compared with high-score reviews",
      x = "Word",
      y = "Low-score association"
    ) +
    theme_premium()

  show_plot_for_interactive_use(p_low_terms)
  ggsave(
    file.path(figures_dir, "aspect_low_score_key_terms.png"),
    plot = p_low_terms,
    width = 11,
    height = 8,
    dpi = 300
  )
} else {
  save_placeholder_plot(
    "aspect_low_score_key_terms.png",
    "Words Most Associated with Low Aspect Scores",
    "This sample does not have enough low-score review text to rank words.",
    width = 11,
    height = 8
  )
}

# Repeat the same idea for two-word phrases. Phrases are often easier for
# non-technical readers to interpret than individual words.
plot_key_phrases <- aspect_text_key_phrases %>%
  group_by(aspect_label) %>%
  slice_max(order_by = log_lift_low_vs_high, n = 5, with_ties = FALSE) %>%
  ungroup() %>%
  mutate(
    term_for_plot = tidytext::reorder_within(term, log_lift_low_vs_high, aspect_label),
    label_text = paste0("low n=", count_low)
  )

if (nrow(plot_key_phrases) > 0) {
  p_low_phrases <- ggplot(
    plot_key_phrases,
    aes(x = term_for_plot, y = log_lift_low_vs_high, fill = term_sentiment)
  ) +
    geom_col(width = 0.72, alpha = 0.88) +
    geom_text(aes(label = label_text), hjust = -0.08, size = 2.4, color = "#2C3E50") +
    coord_flip() +
    facet_wrap(~ aspect_label, scales = "free_y", ncol = 2) +
    tidytext::scale_x_reordered() +
    scale_fill_manual(
      values = c("negative" = "#E76F51", "neutral" = "#457B9D", "positive" = "#2A9D8F"),
      name = "AFINN phrase tone"
    ) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.34))) +
    labs(
      title = "Phrases Most Associated with Low Aspect Scores",
      subtitle = "Bars show smoothed log lift in low-score reviews compared with high-score reviews",
      x = "Phrase",
      y = "Low-score association"
    ) +
    theme_premium()

  show_plot_for_interactive_use(p_low_phrases)
  ggsave(
    file.path(figures_dir, "aspect_low_score_key_phrases.png"),
    plot = p_low_phrases,
    width = 11,
    height = 8,
    dpi = 300
  )
} else {
  save_placeholder_plot(
    "aspect_low_score_key_phrases.png",
    "Phrases Most Associated with Low Aspect Scores",
    "This sample does not have enough low-score review text to rank phrases.",
    width = 11,
    height = 8
  )
}

# This chart keeps only negative AFINN words. It helps identify where low
# aspect scores are connected to explicitly negative language.
negative_heatmap_terms <- aspect_text_key_terms %>%
  filter(term_sentiment == "negative", log_lift_low_vs_high > 0) %>%
  group_by(aspect_label) %>%
  slice_max(order_by = log_lift_low_vs_high, n = 5, with_ties = FALSE) %>%
  ungroup()

if (nrow(negative_heatmap_terms) > 0) {
  p_negative_heatmap <- negative_heatmap_terms %>%
    mutate(
      term = fct_reorder(term, log_lift_low_vs_high, .fun = max),
      heatmap_label = paste0("n=", count_low)
    ) %>%
    ggplot(aes(x = aspect_label, y = term, fill = log_lift_low_vs_high)) +
    geom_tile(color = "white", linewidth = 0.45) +
    geom_text(aes(label = heatmap_label), size = 2.7, color = "#2C3E50") +
    scale_fill_gradient(
      low = "#F7F9F9",
      high = "#C0392B",
      name = "Low-score\nlog lift"
    ) +
    labs(
      title = "Negative Words Associated with Low Aspect Scores",
      subtitle = "Each tile shows a negative AFINN word that appears unusually often in low-score reviews",
      x = "Aspect",
      y = "Negative word"
    ) +
    theme_premium() +
    theme(
      legend.position = "right",
      axis.text.x = element_text(angle = 35, hjust = 1)
    )

  show_plot_for_interactive_use(p_negative_heatmap)
  ggsave(
    file.path(figures_dir, "aspect_negative_term_heatmap.png"),
    plot = p_negative_heatmap,
    width = 10,
    height = 7,
    dpi = 300
  )
} else {
  save_placeholder_plot(
    "aspect_negative_term_heatmap.png",
    "Negative Words Associated with Low Aspect Scores",
    "No negative terms had positive low-score lift in this sample.",
    width = 10,
    height = 7
  )
}

# Count each mismatch type by aspect so reviewers can see where manual reading
# is most needed.
mismatch_counts <- aspect_text_mismatches %>%
  separate_rows(mismatch_types, sep = "; ") %>%
  count(aspect, mismatch_types, name = "review_count")

if (nrow(mismatch_counts) > 0) {
  # Sort aspects by their total number of mismatch cases so the chart has a
  # stable and meaningful order.
  mismatch_aspect_order <- mismatch_counts %>%
    group_by(aspect) %>%
    summarise(total_cases = sum(review_count), .groups = "drop") %>%
    arrange(total_cases) %>%
    pull(aspect)

  mismatch_counts <- mismatch_counts %>%
    mutate(
      aspect = factor(aspect, levels = mismatch_aspect_order),
      mismatch_type_label = str_wrap(mismatch_types, width = 34)
    )

  p_mismatches <- ggplot(
    mismatch_counts,
    aes(x = aspect, y = review_count, fill = mismatch_type_label)
  ) +
    geom_col(width = 0.72, alpha = 0.88) +
    coord_flip() +
    scale_fill_brewer(
      palette = "Set2",
      name = "Mismatch type",
      guide = guide_legend(nrow = 2, byrow = TRUE)
    ) +
    labs(
      title = "Aspect-Text Mismatch Cases for Qualitative Review",
      subtitle = "Counts identify reviews where ratings and text signals do not fully agree",
      x = "Aspect",
      y = "Review-aspect cases"
    ) +
    theme_premium() +
    theme(
      legend.text = element_text(size = 8),
      plot.margin = margin(10, 18, 10, 18)
    )

  show_plot_for_interactive_use(p_mismatches)
  ggsave(
    file.path(figures_dir, "aspect_text_mismatch_counts.png"),
    plot = p_mismatches,
    width = 11,
    height = 7,
    dpi = 300
  )
} else {
  save_placeholder_plot(
    "aspect_text_mismatch_counts.png",
    "Aspect-Text Mismatch Cases for Qualitative Review",
    "This sample did not produce any aspect-text mismatch cases.",
    width = 11,
    height = 7
  )
}

cat("Aspect text analysis complete!\n")
cat("- ", file.path(reports_dir, "aspect_text_alignment.csv"), "\n", sep = "")
cat("- ", file.path(reports_dir, "aspect_text_band_summary.csv"), "\n", sep = "")
cat("- ", file.path(reports_dir, "aspect_text_key_terms.csv"), "\n", sep = "")
cat("- ", file.path(reports_dir, "aspect_text_key_phrases.csv"), "\n", sep = "")
cat("- ", file.path(reports_dir, "aspect_text_mismatches.csv"), "\n", sep = "")
cat("- ", file.path(reports_dir, "aspect_qualitative_examples.csv"), "\n", sep = "")
cat("- ", file.path(figures_dir, "aspect_sentiment_by_rating_boxplot.png"), "\n", sep = "")
cat("- ", file.path(figures_dir, "aspect_low_score_key_terms.png"), "\n", sep = "")
cat("- ", file.path(figures_dir, "aspect_low_score_key_phrases.png"), "\n", sep = "")
cat("- ", file.path(figures_dir, "aspect_negative_term_heatmap.png"), "\n", sep = "")
cat("- ", file.path(figures_dir, "aspect_text_mismatch_counts.png"), "\n", sep = "")